# Multivariate calibration: rank and PIT histograms

Produces the calibration figures of the paper: Figure 5 (the cross-dataset grid of LorD, SB
and SCORE, §7 below) and, in Appendix H.3, Figure 11 (marginal, location and scale PIT ECDFs,
`calibration_grid_calibration.pdf`) and Figure 12 (variogram pre-rank per lag,
`calibration_grid_dependence.pdf`).

Pre-rank diagnostics of Allen, Ziegel & Ginsbourger, *Assessing the calibration of
multivariate probabilistic forecasts* (QJRMS 2024), read back from the results tree's
`<experiment>/calibration/` folders (written by `src/evaluation/calibration.py`).

A pre-rank $\rho:\mathbb{R}^d\to\mathbb{R}$ collapses a forecast vector to a scalar; the rank of
$\rho(y)$ among $\rho(x_1),\dots,\rho(x_M)$ for members $x_m\sim F$ is uniform whenever $F$ is
auto-calibrated, and each choice of $\rho$ targets a different aspect:

| diagnostic | pre-rank | what a deviation means |
|---|---|---|
| **marginal** | $\Phi((y_i-\mu_i)/\sigma_i)$ per coordinate | U-shaped = under-dispersed marginals, humped = over-dispersed |
| **location** | $\rho_{\mathrm{loc}}(x)=\bar{x}$ | the *aggregate* is mis-located or mis-scaled |
| **scale** | $\rho_{\mathrm{sc}}(x)=\frac{1}{d}\sum_j (x_j-\bar{x})^2$ | the overall spread of the field is wrong |
| **dependence** | $\rho_{\mathrm{dep}}(x;h)=-\gamma_x(h)/s^2_x$ | the correlation at separation $h$ is wrong |
| **isotropy** | $\rho_{\mathrm{iso}}(x;h)$ (2-D grids only) | the field's directional structure is wrong |

**Everything is on one scale.** Sampled ranks are converted to their mid-rank PIT
$u=(R-\tfrac12)/(M+1)$ and every histogram is normalized so that a *calibrated* forecast is a flat
line at $1.0$, whatever $M$ or the case count is. Time series, EUPPBench and ERA5 use $M=999$;
depth uses $M=199$, because each draw there is a full $426\times560$ field. Nothing here depends
on which — `K` divides both $1000$ and $200$ exactly, so neither rank support straddles a bin
boundary, and the density normalization puts the panels on one scale regardless.

**Every diagnostic is a line, not a bar**: the empirical CDF $F_n(u)$ against $u$, which lies
**on the diagonal** under calibration. Dependence and isotropy overlay one line per lag; marginal,
location and scale are overlaid in a single *calibration* panel instead, one line per diagnostic --
which is why each dataset below needs only two panel figures (three where isotropy applies) rather
than five. Overlaid step histograms would be unreadable at that density anyway; the ECDF is smooth
and needs no binning at all. A curve above the diagonal means the observed pre-rank sits too low
among the members, below it means too high.

**No confidence bands.** A binomial band assumes independent cases, which is false here: adjacent
time-series windows share 95 of their 96 horizon steps, and depth's marginal histogram pools
$\sim2\times10^8$ spatially-correlated coordinates. A nominal band would be invisibly narrow and
every arm would read as catastrophically miscalibrated.

**Columns, one per method family**: Diag, LorD $r{=}1$, Chol (where trained), SB and SCORE,
whose basis is *task-dependent*. A column with no usable arm is drawn as an annotated blank
rather than dropped, so the columns stay aligned across experiments and a gap is legible.

**Every panel is fitted by the kernel score**, so the figures isolate the *parameterization*:
with the objective held fixed, a difference between two panels in a row is a difference of
architecture. The price is coverage, and it is paid in blanks: an arm that exists only at the
log score or only at a different LorD rank is a blank panel rather than an off-objective one.
LorD is $r{=}1$ or blank, and the SB column is filled only where a *kernel-score* SB arm was
trained (every task here). SB on depth is filled but **degenerate** — the ensemble has collapsed
to a point forecast on all five seeds; see section 2. Both rules are one edit away in the config
cell; the diagnostics for the excluded arms are still in the tree.

**Every panel draws the per-seed median.** Every arm carries seeds 1–5. Every seed's ECDF is
evaluated on a common grid and the **pointwise median** is the line — for the rank-based
diagnostics that grid is the stored support $(1,\dots,M{+}1)/(M{+}1)$, so no interpolation
happens at all; only the continuous location PIT has to be brought onto a common grid first.

**Why the median and not the average.** Summing the stored rank histograms is exactly the average
of the per-seed curves, since every seed scores the same test set at the same $M$. An average is
the wrong summary when a population is *bimodal* rather than noisy, and SB can be: its ensemble
collapses to a point forecast on some seeds and not others (e.g. 2 of 5 seeds at illness
$T{=}36$, all five on depth). The average of a collapsed and a healthy seed describes no trained
model; the median is the typical seed. Retraining only the seeds that collapse would be
cherry-picking — every arm uses one fixed seed set.

The median is safe in the two ways that matter. It **keeps the null**: a pointwise median of
monotone functions is monotone, so the line is still a CDF, and every seed's $F_n(u)\to u$ under
calibration, so their median does too — the diagonal means exactly what it meant. And it
**costs nothing where nothing is wrong**: where no seed collapses (e.g. on ETTh), median and
average agree to within $0.01$. What it gives up is the *visibility* of that bimodality, which is
a fact about one baseline and belongs in the text rather than in a line this figure draws.

**A seed is not an independent case, either way.** EUPPBench's five seeds see 730 cases scored
five times by five *correlated* models, not 3650 independent ones. That compounds the
non-independence noted just above, and is the second reason no band is drawn.

**Closed-form vs sampled PIT.** For a Gaussian family the marginal and location diagnostics are
exact normal CDFs. SB has no density, so both are taken as *ranks* against its draws instead —
uniform under calibration either way, but discrete on the $1/(M+1)$ grid. Each file records which
route it used in `pit_estimator`, and `load_pooled` refuses to mix them silently.

**Only the paper's rows are drawn.** Every time-series task appears at its *shortest* horizon
only — $T = 96$, illness $T = 24$ — and the ETT tasks only in their single-channel mode
(`features: S`). The diagnostics of the other horizons and modes are still in the tree
(`univariate/T192`, `multivariate/T{96,192}`, weather `multivariate/T192`, illness
`multivariate/T36`), so a row comes back as a copy of its $T = 96$ cell with `mode=` changed —
each horizon on its own lag ladder, which the loaders already enforce. Of the time series only
the **ETT** rows are exported; illness and weather are drawn below but not written to `figures/`
(`NOT_EXPORTED`), because the paper does not use them.

**The paper takes those rows as two grids.** Every exported row — ETTh1, ETTh2, ETTm1, ETTm2,
ERA5, NYU-BTS and EUPPBench — is stacked into one figure per kind:
`calibration_grid_calibration.pdf` (marginal, mean and scale, one joint legend; Figure 11) and
`calibration_grid_dependence.pdf` (one legend per row, since every dataset has its own lag
ladder; Figure 12). Rows are datasets, labelled where the single-row figures print $F_n(u)$;
columns are the method families. The single-row figures further down are the same panels, drawn
for reference and not written; with the cross-dataset grid (Figure 5), the two grids are the
whole export.

## Configuration

Every knob is in the next cell: figure sizes, bin count, arms, labels, saving.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

if str(Path.cwd()) not in sys.path:          # so `import plot_config` works from anywhere
    sys.path.insert(0, str(Path.cwd()))
import plot_config as pc

PLOT = pc.apply_style()                       # scienceplots + the paper's rcParams
# Every figure cell returns its figure for display, so all of them stay open: 22 by the end
# of a run since the two grids, over the 20 at which pyplot warns on every new figure.
plt.rcParams["figure.max_open_warning"] = 0

# --- where the diagnostics live ---------------------------------------------------------
from src.utils.paths import configured_path
SHARED = configured_path("results_dir")

# --- saving ------------------------------------------------------------------------------
# Off by default. When True, every figure lands in notebooks/figures/ at exactly its paper
# size (pc.save uses no bbox_inches="tight", so the PDF width is the requested column width).
# A row with no diagnostics at all is drawn but never saved -- an all-blank PDF is not a
# figure, it is a placeholder, and it should not accumulate in figures/.
SAVE = True

# Rows that are drawn but never written, whatever SAVE says: of the time series the paper
# uses only the ETT rows (next to depth, EUPPBench, ERA5 and the cross-dataset grid), so
# illness and weather stay on screen for reference and out of figures/.
#
# The rows the paper *does* use reach it as the two grids (`GRID_ROWS`, section "Paper
# figures") rather than one figure per row, so the single-row figures of those datasets are
# drawn for reference and not written either. What is exported is the two grids and the
# cross-dataset figure.
NOT_EXPORTED = (
    "illness_t24_calibration", "illness_t24_dependence",
    "weather_t96_calibration", "weather_t96_dependence",
) + tuple(f"{row}_{kind}"
          for row in ("ts_etth1", "ts_etth2", "ts_ettm1", "ts_ettm2", "depth", "eupp", "era5")
          for kind in ("calibration", "dependence"))

# --- per-figure column width -------------------------------------------------------------
# "single" (3.25 in) or "double" (6.75 in) -- the paper's two text widths. Change any entry
# to re-emit that one figure at the other width; the plotting functions also take an explicit
# `columns=` that overrides this table.
FIG_COLUMNS = {
    "ts_etth1_calibration":    "double",
    "ts_etth1_dependence":     "double",
    "ts_etth2_calibration":    "double",
    "ts_etth2_dependence":     "double",
    "depth_calibration":       "double",
    "depth_dependence":        "double",
    "depth_isotropy":          "double",
    "eupp_calibration":        "double",
    "eupp_dependence":         "double",
    "era5_calibration":        "double",
    "era5_dependence":         "double",
    "era5_isotropy":           "double",
    "illness_t24_calibration": "double",
    "illness_t24_dependence":  "double",
    "weather_t96_calibration": "double",
    "weather_t96_dependence":  "double",
    # The two paper grids: every dataset row stacked, one per kind (`plot_ecdf_grid`).
    "grid_calibration":        "double",
    "grid_dependence":         "double",
    # The cross-dataset grid is designed single-column (3x4 panel: families x diagnostics).
    "cross_dataset_loc_scale": "single",
}

# --- seed aggregation --------------------------------------------------------------------
# How an arm's seeds become the one curve a panel draws. **`"median"`**: every seed's ECDF is
# evaluated on a common grid and the *pointwise median* is drawn.
#
# Not `"pooled"` (summing the stored histograms) because pooling averages, and one arm here is
# bimodal rather than noisy: SB's ensemble can collapse on some seeds and not others -- those
# seeds become point forecasts, ~1-2 % of the correct predictive spread (e.g. illness T36, 2 of
# 5 seeds; depth, 5 of 5). An average of the two populations describes no trained model; the
# median is the typical seed. Retraining only the bad seeds would be cherry-picking -- every
# arm uses one fixed seed set.
#
# It costs nothing where the seeds agree (on ETTh median and pooled agree to within 0.01).
# And it keeps the diagnostic's meaning intact -- the pointwise median of monotone functions
# is monotone, so the drawn curve is still a CDF, and since each seed's F(u) -> u under
# calibration, so does their median. The diagonal is still exactly the null.
#
# What it gives up is the *visibility* of that bimodality: a reader sees the typical seed and
# not that one arm has two modes. That is handled in the text, not by this figure.
# `"pooled"` gives the averaged version; `SEED_AGGREGATION` is read only by
# `ecdf_over_seeds`.
SEED_AGGREGATION = "median"

# --- colors ------------------------------------------------------------------------------
# Every arm in this notebook is kernel-score fitted (see the arms block below), so the fill
# color is constant. The mapping is kept rather than hard-coded: it is what would distinguish
# an off-objective arm if one were added, and blank panels still read their `objective` key.
# Lag/bin lines use the shared sequential ramp, so the color ordering reads as "further apart".
OBJECTIVE_COLOR = {
    "ks":  pc.COLORS["kernel_score"],
    "nll": pc.COLORS["log_score"],
    "es":  pc.COLORS["energy_score"],
}
LAG_CMAP = pc.sequential_cmap()
GUIDE = pc.COLORS["guide"]

# The combined marginal/location/scale panel (`plot_ecdf(..., kind="calibration")`) draws one
# ECDF line per diagnostic rather than one bar histogram per diagnostic, so each needs its own
# fixed color -- distinct from `OBJECTIVE_COLOR` above, which encodes the *training* objective,
# not the diagnostic being plotted.
CALIBRATION_OBJECTIVES = ("marginal", "loc", "scale")
CALIBRATION_COLOR = {
    "marginal": "#3b4cc0",   # blue
    "loc":      "#d6604d",   # red
    "scale":    "#4dac26",   # green
}
CALIBRATION_LABEL = {
    "marginal": "Marginal",
    "loc":      "Mean",
    "scale":    "Scale",
}

# --- the arms ----------------------------------------------------------------------------
# One per method family, in panel order (left to right): the physical-space baselines (Diag,
# LorD, and Chol where it is tractable), SB, and SCORE -- whose basis is task-dependent (the
# graph Fourier transform with a Matern filter on the station network, the spectral diagonal
# core on a grid or a horizon).
#
# **Every arm is fitted by the kernel score.** With the objective held fixed, a difference
# between two panels is a difference of *parameterization*. It costs coverage -- an arm that
# exists only at the log score is a blank panel rather than an off-objective one -- and the
# two rules applied are strict:
#
#   * LorD is `r=1` or blank; a different rank is never substituted into the column.
#   * The SB column is filled wherever a kernel-score SB arm was trained (every task here).
#     It is fitted by the fair *kernel score* -- the same estimator the reported metric uses,
#     which is what lets it into an all-KS row.
#
# SB's panel is the one place in the notebook where the marginal and location histograms are
# **sampled ranks** rather than closed-form normal CDFs (a sample-based model has no mu and
# sigma to evaluate), which is what `load_pooled`'s `pit_estimator` guard is there to keep
# from being pooled with the closed-form panels.
#
# `objective` picks the fill color; labels are LaTeX-safe (the `science` style runs
# text.usetex, so no raw slugs and no accented characters in anything that reaches a title).
#
# `seeds` is a list: every panel pools *all* of them (see `load_pooled`). An arm with
# `slug=None` is one this project has no diagnostics for; its `note` is what the blank panel
# prints, so a gap in a row is legible instead of looking like a plotting bug.

# Every arm of every row is on seeds 1-5.
EUPP_SEEDS = [1, 2, 3, 4, 5]      # the station post-processing runs
TS_SEEDS = [1, 2, 3, 4, 5]        # the ETTh T96/T192 runs

# Depth uses the Eigen-cropped `nyu_bts_crop` event (426x560). The seed lists are written per
# arm rather than as one shared list, so a row whose arms differ in seed coverage stays a
# one-line edit here.
DEPTH_SEEDS = [1, 2, 3, 4, 5]
DEPTH_LORA_SEEDS = [1, 2, 3, 4, 5]
ERA5_SEEDS = [1, 2, 3, 4, 5]
ERA5_LORA_SEEDS = [1, 2, 3, 4, 5]
# SB is the one arm where a re-run of the diagnostics is not cheap: on ERA5 its sampler needs
# one backbone pass per draw chunk (125 passes x 1092 cases at `--draw-chunk 8`), i.e. tens of
# minutes per seed against ~1.5 min for every closed-form arm. `needs_refresh` keys the skip on
# `schema_version`, so an interrupted run resumes without recomputing finished seeds. Never pass
# `--overwrite` to resume this arm.

# The one **five**-panel row besides ETTh. It reads left to right as increasing covariance
# generality inside physical space -- Diag, LorD r=1, Chol -- then SB, then SCORE.
#
# `phys_cholesky_kernel_score` (Chol) is the unstructured ceiling: a full Sigma = L Lᵀ,
# tractable here because V = 115. It belongs on this row precisely because it bounds what *any*
# covariance parameterization can do on this task, so a gap between it and SCORE is a gap in
# the parameterization rather than in the fitting.
#
# `_s1crps_dense_kernel_score`: stage 1 is CRPS, and stage 2 scores the predictive covariance
# Sigma_P against y - mu directly. On the localized graph basis the renormalization by the
# implied marginal variances makes D~ depend on the stage-2 parameter, so scoring the core
# against a fixed standardized target would not be a proper score for Sigma_P.
#
# The arms the cross-dataset figure reuses are bound to names here rather than reached by
# index, so a change of panel order cannot silently put the wrong arm in that figure.
EUPP_LORA = dict(slug="phys_lora_r1_kernel_score",   seeds=EUPP_SEEDS, objective="ks",
                 label=r"LorD $r{=}1$",          short="LorD")
EUPP_OURS = dict(slug="whitened_graph_matern_s1crps_dense_kernel_score",
                                                     seeds=EUPP_SEEDS, objective="ks",
                 label="SCORE: graph Matern",    short="SCORE")
EUPP_GEN  = dict(slug="generative_kernel_score",     seeds=EUPP_SEEDS, objective="ks",
                 label="SB",                     short="SB")

EUPP_ARMS = [
    dict(slug="phys_diag_kernel_score",           seeds=EUPP_SEEDS,      objective="ks",
         label="Diag",                        short="Diag"),
    EUPP_LORA,
    dict(slug="phys_cholesky_kernel_score",       seeds=EUPP_SEEDS,      objective="ks",
         label="Chol",                        short="Chol"),
    EUPP_GEN,
    EUPP_OURS,
]

# **ETTh** reads `univariate/T96` and `univariate/T192`, seeds 1-5, `data_seed` 42.
#
# Why the univariate mode: `phys_cholesky` (Chol) -- the unstructured ceiling -- is trained
# only at `features: S`, where V = pred_len keeps the dense factor affordable, so only this
# mode fills the five-panel row. The geometry: `seq_len: 336`, `target: OT`, 2785 test cases
# at d=96 and 2689 at d=192. The `ett_hour` split is chronological and takes no seed.
#
# SCORE is the rank-free core, `whitened_spectral_diag_s1crps_kernel_score` at `rank: 0` --
# the same slug and label as the depth and ERA5 rows -- and the LorD column is
# `phys_lora_r1_kernel_score`, as everywhere else. The SB column is filled on both horizons.
#
# One list serves both horizons: the slugs are identical and only `mode` differs, so T96 and
# T192 cannot drift apart in arm identity. Order matches the EUPPBench row -- Diag, LorD,
# Chol, SB, SCORE -- so the two five-panel rows read the same left to right.
#
# The slots the cross-dataset figure reuses are bound to names, not indices, so a change of
# panel count cannot silently swap the arm drawn there.
TS_LORA = dict(slug="phys_lora_r1_kernel_score",     seeds=TS_SEEDS, objective="ks",
               label=r"LorD $r{=}1$",            short="LorD")
TS_OURS = dict(slug="whitened_spectral_diag_s1crps_kernel_score",
                                                     seeds=TS_SEEDS, objective="ks",
               label="SCORE: spectral diag",     short="SCORE")
TS_GEN  = dict(slug="generative_kernel_score",       seeds=TS_SEEDS, objective="ks",
               label="SB",                       short="SB")

TS_ARMS = [
    dict(slug="phys_diag_kernel_score",              seeds=TS_SEEDS, objective="ks",
         label="Diag",                           short="Diag"),
    TS_LORA,
    dict(slug="phys_cholesky_kernel_score",          seeds=TS_SEEDS, objective="ks",
         label="Chol",                           short="Chol"),
    TS_GEN,
    TS_OURS,
]

# **ETTm** (univariate T96/T192, seeds 1-5) is the ETTh row without its Chol panel: Chol is
# not in the ETTm grid. Filtered by slug from `TS_ARMS`, so the two rows cannot drift apart in
# labels or seeds.
ETTM_ARMS = [a for a in TS_ARMS if a["slug"] != "phys_cholesky_kernel_score"]

# Depth: Diag, LorD r=1, SB and SCORE, at five seeds each, at one objective.
# `phys_lora_r16_kernel_score` is also diagnosed at all five seeds and stays out only because
# of the rank rule. SB's diagnostics need a 48 GB card: the un-chunkable `model(x)` call in
# calibration.py wants ~27 GiB at V = 238,560 (see the header of runs/calibration.sh).
#
# **Read the SB panel as a collapse, not as a calibration.** All five seeds score
# crps/mae = 0.9997 -- the CRPS of a point forecast is its MAE -- so the members are
# effectively identical and the ensemble has no spread. The marginal PIT says the same thing
# independently: ~0.1% of the mass lands in the central 90%, against 88% for Diag and 45%
# for SCORE, because the observation is below every draw or above every draw and essentially
# never between them. The panel is drawn rather than suppressed: it is a true diagnostic of
# the trained model (the same failure mode appears on illness T36, 2 of 5 seeds).
DEPTH_ARMS = [
    dict(slug="phys_diag_kernel_score",                        seeds=DEPTH_SEEDS, objective="ks",
         label="Diag",                                     short="Diag"),
    dict(slug="phys_lora_r1_kernel_score",                     seeds=DEPTH_LORA_SEEDS,
                                                                                  objective="ks",
         label=r"LorD $r{=}1$",                            short="LorD"),
    dict(slug="generative_kernel_score",                       seeds=DEPTH_SEEDS, objective="ks",
         label="SB",                                       short="SB"),
    dict(slug="whitened_spectral_diag_s1crps_kernel_score",    seeds=DEPTH_SEEDS, objective="ks",
         label="SCORE: spectral diag",                     short="SCORE"),
]

# ERA5 needs **two** datamodule overrides, and only the first is about file locations:
#   --data-path <era5 data dir>/utc12/   (if the configs name a path that does not resolve)
#   --dm-override statistics_path=<...>/era5_statistics_train1990-2014.npy
# The second one is a *different normalization*, not another copy of the same file: the
# runs standardize with statistics computed on the training years only, while the
# datamodule's default (<data_path>/era5_statistics.npy) was computed over 1990-2022,
# validation and test years included. Passing only --data-path raises FileNotFoundError
# here, which is the good case -- silently falling back to the wrong statistics would score
# these checkpoints on a normalization they were never fitted for.
#
# All four panels are filled, at the same five seeds. SCORE is the two-stage fit whose
# stage 1 is CRPS and stage 2 the kernel score. The kernel-score LorD ranks 2, 4 and 16 are
# diagnosed too; the rank rule keeps them out of this figure.
#
# SB is by far the most expensive arm to diagnose here: its sampler needs one backbone pass
# per draw chunk, so on this grid (V = 35,200, 1092 cases) it is ~25 min per seed at
# `--draw-chunk 8` against ~1.5 min for the closed-form arms. Its marginal and mean pre-ranks
# are *sampled* rather than closed form, as they are for EUPPBench's.
ERA5_ARMS = [
    dict(slug="phys_diag_kernel_score",                     seeds=ERA5_SEEDS,      objective="ks",
         label="Diag",                                  short="Diag"),
    dict(slug="phys_lora_r1_kernel_score",                  seeds=ERA5_LORA_SEEDS, objective="ks",
         label=r"LorD $r{=}1$",                         short="LorD"),
    dict(slug="generative_kernel_score",                    seeds=ERA5_SEEDS,      objective="ks",
         label="SB",                                    short="SB"),
    dict(slug="whitened_spectral_diag_s1crps_kernel_score", seeds=ERA5_SEEDS,      objective="ks",
         label="SCORE: spectral diag",                  short="SCORE"),
]
# Illness and weather are the two *multivariate-only* benchmarks: no univariate counterpart
# exists, so unlike ETTh they cannot sidestep the channel question (see sections 5 and 6).
#
# Both are filled rows on one arm list (seeds 1-5; illness at T24/T36, weather at T96/T192):
# Diag, LorD r=1, SB and the rank-free SCORE core
# `whitened_spectral_diag_s1crps_kernel_score` -- the same slugs the depth and ERA5 rows use.
ILLNESS_SEEDS = [1, 2, 3, 4, 5]
WEATHER_SEEDS = [1, 2, 3, 4, 5]


#: Illness: the same four slots as the depth and ERA5 rows, all four filled, at five seeds.
ILLNESS_ARMS = [
    dict(slug="phys_diag_kernel_score",                     seeds=ILLNESS_SEEDS, objective="ks",
         label="Diag",                                  short="Diag"),
    dict(slug="phys_lora_r1_kernel_score",                  seeds=ILLNESS_SEEDS, objective="ks",
         label=r"LorD $r{=}1$",                         short="LorD"),
    dict(slug="generative_kernel_score",                    seeds=ILLNESS_SEEDS, objective="ks",
         label="SB",                                    short="SB"),
    dict(slug="whitened_spectral_diag_s1crps_kernel_score", seeds=ILLNESS_SEEDS, objective="ks",
         label="SCORE: spectral diag",                  short="SCORE"),
]
WEATHER_ARMS = [dict(a, seeds=WEATHER_SEEDS) for a in ILLNESS_ARMS]

# --- narrow-figure adjustments -----------------------------------------------------------
# plot_config's font sizes are absolute points, so they do NOT shrink when a figure is
# emitted at single-column width: a row of four 25-character panel titles that is fine at
# 6.75 in collides badly at 3.25 in. Below `NARROW_BELOW` inches of *figure* width the fonts
# step down to `NARROW_RC`; at or above it every figure keeps plot_config's sizes, whatever
# its panel count. That last clause is the point of the split below: two figures printed at
# the same width belong in the paper with the same title, label and tick sizes, and a
# four-panel row sitting next to a five-panel row in smaller type is a difference the reader
# can see and cannot interpret.
NARROW_BELOW = 4.5

# What actually *collides*, though, is the width of one *panel*, not of the figure: at double
# width a five-panel row leaves 1.35 in per panel against 1.69 in for a four-panel row, and the
# long titles overrun the figure edge. So the long/`short` label switch is made per panel --
# and only that switch; the font step-down stays tied to the figure width. Any threshold in
# (1.35, 1.69] leaves every four-panel row on the long labels and sends the five-panel rows to
# the short ones.
NARROW_PANEL_BELOW = 1.5

# Shape of one ECDF panel: box height over box width, the same in every row. `plot_ecdf`
# fits the *figure height* to it after layout, so a four-panel row (no Chol) and a
# five-panel row draw their panels -- and the diagonal -- at one slope. The price is that
# the rows no longer share one height: title, x label, ticks and legend cost the same
# ~0.9 in either way, and only the panel scales with its (wider) four-panel width.
PANEL_ASPECT = 0.4

# Height of a row as a fraction of the figure width, used only where the fit above *starts*
# (and the final height if a caller passes `panel_aspect=None`).
ROW_ASPECT = 0.21
NARROW_RC = {
    "font.size": 7.0,
    "axes.titlesize": 6.5,
    "axes.labelsize": 7.0,
    "xtick.labelsize": 6.0,
    "ytick.labelsize": 6.0,
    "legend.fontsize": 6.0,
}

# Depth stores 5 variogram directions; the two diagonals sit at h*sqrt(2), not h, so they
# cannot share a lag axis with the axial ones. `axial_mean` is the right summary on a
# square-pixel grid. Time series and EUPPBench have a single direction each.
DEPTH_DEP_DIRECTION = "axial_mean"

# ERA5 is a lat-lon grid, so its pixel lags were converted to km and binned (see section 4).
# Once every direction is on a common distance axis they can be pooled, which is what
# `pooled` is: the pair-weighted variogram of all four directions per bin, not a mean of
# ratios. Isotropy still reads the four directions separately -- at matched km.
ERA5_DEP_DIRECTION = "pooled"

# --- the cross-dataset figure ------------------------------------------------------------
# One line per dataset, so this figure asks whether a parameterization behaves the same way
# across tasks. ETTh1/ETTh2 are a light/dark pair on purpose: same benchmark family.
DATASET_COLORS = {
    "ETTh1":     "#1f78b4",
    "ETTh2":     "#a6cee3",
    "NYU-BTS":   "#33a02c",
    "EUPPBench": "#ff7f00",
    "ERA5":      "#6a3d9a",
}

# The low-rank / SCORE slots are named rather than indexed: a positional reference into the
# lists above would break *silently* on a change of panel order -- the figure would still
# render, with the wrong arm in it. Both slots reuse the per-experiment arms, so the r=1 rule
# reaches this figure too. Every row carries all five datasets (ETTh1, ETTh2, NYU-BTS,
# EUPPBench, ERA5); `has_data` still guards them, and the legend is built across every panel
# so a dataset present in only one row still gets an entry.
# `generative` (SB) is the third row of the grid. NYU-BTS is the degenerate one -- its
# ensemble has collapsed to a point forecast on every seed (see the DEPTH_ARMS note above), so
# its curve hugs the panel edges rather than the diagonal. That is the arm's real behaviour
# and is left on the page.
#
# `dep_direction` is the third column's variogram direction, and it is per dataset because
# the geometries differ: a 1-D series has only `axis0`, depth pools its four grid directions
# into `axial_mean`, ERA5's lat-lon lags are converted to km and `pooled`, and the station
# network has no directions at all (`distance_bin`). These are the same values the
# per-experiment dependence figures above are drawn with.
CROSS_DATASETS = [
    dict(label="ETTh1", experiment="time_series", dataset="etth1",
         mode="univariate/T96",
         lora=TS_LORA, whitened=TS_OURS, generative=TS_GEN,
         dep_direction="axis0"),
    dict(label="ETTh2", experiment="time_series", dataset="etth2",
         mode="univariate/T96",
         lora=TS_LORA, whitened=TS_OURS, generative=TS_GEN,
         dep_direction="axis0"),
    dict(label="NYU-BTS", experiment="depth", dataset="nyu_bts", mode=None,
         lora=DEPTH_ARMS[1], whitened=DEPTH_ARMS[3], generative=DEPTH_ARMS[2],
         dep_direction=DEPTH_DEP_DIRECTION),
    dict(label="EUPPBench", experiment="station_postprocessing", dataset="euppbench", mode=None,
         lora=EUPP_LORA, whitened=EUPP_OURS, generative=EUPP_GEN,
         dep_direction="distance_bin"),
    dict(label="ERA5", experiment="surface_temperature", dataset="era5", mode=None,
         lora=ERA5_ARMS[1], whitened=ERA5_ARMS[3], generative=ERA5_ARMS[2],
         dep_direction=ERA5_DEP_DIRECTION),
]

# --- the two paper grids -----------------------------------------------------------------
# The paper takes the per-dataset rows as **two** figures (Figures 11 and 12): every
# calibration row stacked into one grid, every dependence row into another (`plot_ecdf_grid`,
# section "Paper figures"). Rows are datasets, top to bottom the time series first, then ERA5,
# depth and EUPPBench; columns are method families, so a family keeps one column even where a
# row has no arm for it.
#
# The rows reuse the per-dataset arm lists above, so the grids cannot drift from the
# single-row figures -- same slugs, same seeds, same mode, same variogram direction. Only the
# rows the paper uses are here: illness and weather are not (see `NOT_EXPORTED`).
GRID_ROWS = [
    dict(label="ETTh1",     experiment="time_series", dataset="etth1",
         mode="univariate/T96", arms=TS_ARMS,   direction="axis0"),
    dict(label="ETTh2",     experiment="time_series", dataset="etth2",
         mode="univariate/T96", arms=TS_ARMS,   direction="axis0"),
    dict(label="ETTm1",     experiment="time_series", dataset="ettm1",
         mode="univariate/T96", arms=ETTM_ARMS, direction="axis0"),
    dict(label="ETTm2",     experiment="time_series", dataset="ettm2",
         mode="univariate/T96", arms=ETTM_ARMS, direction="axis0"),
    dict(label="ERA5",      experiment="surface_temperature", dataset="era5",
         mode=None, arms=ERA5_ARMS,  direction=ERA5_DEP_DIRECTION),
    dict(label="NYU-BTS",   experiment="depth", dataset="nyu_bts",
         mode=None, arms=DEPTH_ARMS, direction=DEPTH_DEP_DIRECTION),
    dict(label="EUPPBench", experiment="station_postprocessing", dataset="euppbench",
         mode=None, arms=EUPP_ARMS,  direction="distance_bin"),
]

# The grid's columns, matched against each arm's `short` name -- which is also the column
# title, so the titles are the method names and nothing row-specific: SCORE's basis differs
# by row (graph Fourier/Matern on EUPPBench, spectral diagonal elsewhere) and belongs in the
# caption. Chol exists only on ETTh and EUPPBench, so the other rows draw a blank there with
# this note.
GRID_COLUMNS = ("Diag", "LorD", "Chol", "SB", "SCORE")
GRID_BLANK_NOTE = "not trained"

# The dependence grid's per-row legends, in inches: the gap between a row's lowest
# decoration (the panel frame, or the x labels on the bottom row) and its legend, and the
# extra room left below the legend before the next row's layout gap. Small above and
# larger below, so a legend reads as the caption of the row it sits under rather than as the
# header of the next one -- which is what a legend centred in the gap looked like.
ROW_LEGEND_GAP = (0.05, 0.03)


## Loading and reduction helpers

In [ ]:
_CACHE = {}

#: Keys that define the *axes* of a file rather than its contents. Two seeds may only be
#: pooled if they agree on all of them -- see `load_pooled`.
_AXIS_KEYS = ("dep_kind", "dep_directions", "dep_bin_edges_km", "dep_lags", "n_draws")

#: Rank/PIT histograms, which pool by summing counts. Not every file has all of them
#: (`iso_hist_counts` is 2-D-grid only).
_POOLED_HISTS = ("marg_hist_counts", "var_hist_counts", "dep_hist_counts", "iso_hist_counts")


def has_data(arm):
    """Whether an arm has any diagnostics to plot (a `slug=None` arm is a blank panel)."""
    return arm.get("slug") is not None and bool(arm.get("seeds"))


def arm_path(experiment, dataset, arm, seed, mode=None):
    """`<experiment>/calibration/<dataset>/[<mode>/]<slug>/calibration_seed<N>.npz`."""
    parts = [SHARED, experiment, "calibration", dataset]
    if mode is not None:
        parts.append(mode)
    parts += [arm["slug"], f"calibration_seed{seed}.npz"]
    return Path(*parts)


def load(experiment, dataset, arm, seed, mode=None):
    """Every array of one arm-seed's npz, cached. Raises with the path if it is missing."""
    path = arm_path(experiment, dataset, arm, seed, mode)
    key = str(path)
    if key not in _CACHE:
        if not path.exists():
            raise FileNotFoundError(
                f"no diagnostics for {arm['slug']} seed{seed}: {path}\n"
                f"    run: python -m src.evaluation.calibration <seed folder> --device cuda"
            )
        with np.load(path, allow_pickle=False) as z:
            _CACHE[key] = {k: z[k] for k in z.files}
    return _CACHE[key]


def pending(experiment, dataset, arms, mode=None):
    """`arms` with every arm whose seed files are not all written drawn as a labelled blank.

    For rows whose diagnostics are still being written -- the sampler pass of 2026-09-28 runs
    on the cluster after the closed-form arms were written locally. The panel names what is
    missing instead of the cell raising, and the same cell draws the real arm once every
    seed's file exists. Deliberately *not* folded into `plot_ecdf`: everywhere else a missing
    file is an error, and only an absent file is forgiven here -- a file that exists but
    disagrees on its axes still raises in `load_pooled`.
    """
    out = []
    for arm in arms:
        if not has_data(arm):
            out.append(arm)
            continue
        missing = [s for s in arm["seeds"]
                   if not arm_path(experiment, dataset, arm, s, mode).exists()]
        out.append(dict(arm, slug=None, seeds=[],
                        note=f"pending: {len(missing)} of {len(arm['seeds'])} seeds")
                   if missing else arm)
    return out


def load_pooled(experiment, dataset, arm, mode=None):
    """Every seed of one arm, pooled into a single npz-shaped dict.

    All seeds of an arm score the *same* test set with the same member count, so summing the
    stored histograms is arithmetically identical to averaging the per-seed densities bin by
    bin. That average is **no longer what the panels draw** -- see `SEED_AGGREGATION`, which
    makes them the pointwise *median* over seeds -- but the summed arrays are kept because
    they are what `dep_counts` / `iso_counts` / `lag_labels` read to work out how many lags
    there are and what to call them, and because switching back is then one constant.

    `per_seed` carries the individual dicts, each npz-shaped, so any reducer below can be
    re-applied one seed at a time. That is exactly what `ecdf_over_seeds` does.

    The result reuses the npz's own key names, so every reducer below (`objective_ecdf`,
    `dep_ecdf_pooled`, `dep_counts`, `iso_counts`, `lag_labels`) works on it unchanged --
    and on each entry of `per_seed` unchanged too, which is what makes the median a wrapper
    rather than a rewrite.
    """
    if not has_data(arm):
        raise ValueError(f"arm {arm['label']!r} has no diagnostics to pool")
    seeds = list(arm["seeds"])
    per_seed = [load(experiment, dataset, arm, s, mode) for s in seeds]
    first = per_seed[0]

    # Guard the axes *before* summing. A file written with different `--bin-edges-km` or
    # `--lags` bins the same quantity differently, and nothing downstream would notice: the
    # arrays still have the same shape, they still add, `schema_version` still checks out,
    # and only the legend would be quietly wrong.
    for seed, z in zip(seeds[1:], per_seed[1:]):
        for key in _AXIS_KEYS:
            if (key in first) != (key in z):
                raise ValueError(
                    f"{arm['slug']}: seed{seeds[0]} and seed{seed} disagree on whether "
                    f"{key!r} is present; they were produced by different code versions."
                )
            if key in first and not np.array_equal(first[key], z[key]):
                raise ValueError(
                    f"{arm['slug']}: seed{seeds[0]} and seed{seed} disagree on {key!r}\n"
                    f"    seed{seeds[0]}: {first[key]}\n"
                    f"    seed{seed}: {z[key]}\n"
                    f"    They cannot be pooled. Regenerate both on one axis:\n"
                    f"    python -m src.evaluation.calibration <seed folders> "
                    f"--bin-edges-km 0,50,100,200,300,500,inf --overwrite\n"
                    f"    (--overwrite is required: the skip check keys on schema_version, "
                    f"not on the axis.)"
                )

    pooled = {k: first[k] for k in _AXIS_KEYS if k in first}
    pooled["marg_hist_edges"] = first["marg_hist_edges"]
    for key in _POOLED_HISTS:
        if key in first:
            pooled[key] = np.sum([z[key] for z in per_seed], axis=0)
    # Concatenated along *cases*, not summed over channels -- `objective_ecdf`'s `loc`
    # branch ravels this, which pools a multivariate run's channels along with its seeds.
    pooled["mean_pit"] = np.concatenate([z["mean_pit"] for z in per_seed], axis=0)
    pooled["event_shape"] = first["event_shape"] if "event_shape" in first else None

    # Whether the marginal/mean PIT is a closed-form normal CDF or a rank against draws.
    # SB is the sampled one; every Gaussian family is closed form. Files written without
    # the key are all closed form (see calibration.py).
    estimators = {str(z.get("pit_estimator", "closed_form")) for z in per_seed}
    pooled["pit_estimator"] = np.array(
        estimators.pop() if len(estimators) == 1 else "mixed"
    )
    pooled["n_seeds"] = np.int32(len(seeds))
    pooled["n_cases"] = np.int32(sum(int(z["n_cases"]) for z in per_seed))
    # The only non-array entry, and the one `ecdf_over_seeds` needs. Kept on the same dict
    # rather than returned beside it so that every `z` already threaded through the plotting
    # code carries its own seeds with it and no call site has to grow an argument.
    pooled["per_seed"] = per_seed
    return pooled


def ecdf_from_counts(counts):
    """`(u, F_n(u))` from rank counts over 1..M+1 -- on the diagonal under calibration.

    Built from the stored histogram rather than the per-case ranks, so it is exact and needs
    no binning parameter. Undefined cases (rank 0) are already excluded upstream by
    `_Accumulator.rank_histogram`, so they cannot fake a left-edge spike.
    """
    counts = np.asarray(counts, dtype=np.float64).ravel()
    m1 = counts.size                       # M + 1 possible ranks
    u = np.arange(1, m1 + 1) / m1
    total = counts.sum()
    if total <= 0:
        return u, np.full(m1, np.nan)
    return u, np.cumsum(counts) / total


#: Grid the per-seed ECDFs are compared on when they do not already share one. The
#: rank-based objectives do share one -- `(1..M+1)/(M+1)`, and `n_draws` is an axis key, so
#: two poolable seeds cannot disagree on M -- and there the median is taken at the stored
#: support with no interpolation at all. Only the location PIT is continuous, so its ECDF
#: jumps at its own sample values and the seeds have to be brought onto a common grid first.
#: 1001 points is finer than the line width at every figure size here.
_COMMON_GRID = np.linspace(0.0, 1.0, 1001)


def _step_eval(u, f, grid):
    """A right-continuous ECDF `(u, f)` evaluated at `grid`.

    `np.interp` would be wrong here: an ECDF is a step function and interpolating it
    linearly invents mass between the steps. `searchsorted` takes the value of the last step
    at or below each grid point, and 0 below the first, which is the ECDF's own definition.
    """
    idx = np.searchsorted(np.asarray(u, dtype=np.float64), grid, side="right") - 1
    f = np.asarray(f, dtype=np.float64)
    return np.where(idx >= 0, f[np.clip(idx, 0, None)], 0.0)


def ecdf_over_seeds(z, reducer, *args):
    """`(u, F(u))` of one diagnostic, aggregated over an arm's seeds.

    `reducer` is any of the `(z, ...) -> (u, F)` functions below, applied to each per-seed
    dict rather than to the summed one. Under `SEED_AGGREGATION == "median"` the result is
    the **pointwise median** of those curves; under `"pooled"` it is the reducer applied to
    the summed histograms, i.e. what this notebook drew before 2026-09-08.

    Two properties make the median a safe swap rather than a new diagnostic. It is still a
    CDF -- a pointwise median of monotone functions is monotone -- and it still has the
    diagonal as its null, because every seed's `F(u)` converges to `u` under calibration and
    so therefore does their median. What changes is only which seed the curve describes: the
    typical one instead of a mixture of all five.
    """
    per_seed = z.get("per_seed") if isinstance(z, dict) else None
    if SEED_AGGREGATION != "median" or not per_seed:
        return reducer(z, *args)
    curves = [reducer(s, *args) for s in per_seed]
    grids = [np.asarray(u, dtype=np.float64) for u, _ in curves]
    if all(g.shape == grids[0].shape and np.allclose(g, grids[0]) for g in grids):
        grid = grids[0]                      # rank histograms: exact, no interpolation
    else:
        grid = _COMMON_GRID                  # the continuous location PIT
    stack = np.vstack([_step_eval(u, f, grid) for u, f in curves])
    return grid, np.median(stack, axis=0)


def lag_ecdf(z, kind, direction, j):
    """`(u, F)` of one lag / distance bin, so that `ecdf_over_seeds` can reduce it per seed."""
    counts = iso_counts(z) if kind == "isotropy" else dep_counts(z, direction)
    return ecdf_from_counts(counts[:, j])


def objective_ecdf(z, objective):
    """`(u, F_n(u))` for one pre-rank objective: marginal, location (`loc`) or scale.

    `loc` is the pooled PIT sample, so its ECDF is the sorted sample; the other two are the
    ECDF of the stored exact rank histograms.
    """
    if objective == "loc":
        v = np.asarray(z["mean_pit"], dtype=np.float64).ravel()
        v = np.sort(v[np.isfinite(v)])
        return v, np.arange(1, v.size + 1) / v.size
    if objective == "scale":
        return ecdf_from_counts(np.asarray(z["var_hist_counts"]).sum(axis=1))
    if objective == "marginal":
        return ecdf_from_counts(z["marg_hist_counts"])
    raise ValueError(f"unknown objective {objective!r}")


def dep_ecdf_pooled(z, direction):
    """ECDF of the dependence pre-rank, pooled over every lag / distance bin.

    The cross-dataset grid needs *one* curve per dataset, but the dependence diagnostic is
    stored per lag and the ladders are not comparable across tasks -- a grid lag is a pixel
    offset, a station pair is a distance in km. Pooling is what makes that panel well posed
    instead of a comparison of different axes.

    It is a legitimate reduction, not a convenience: each lag's rank is uniform under
    calibration, so any mixture of them is uniform too, and the pooled curve leaves the
    diagonal exactly when the arm mis-states its dependence. What it gives up is *where* --
    the per-experiment dependence figures keep the lags separate for that.
    """
    return ecdf_from_counts(dep_counts(z, direction).sum(axis=1))


def dep_counts(z, direction):
    """`(M+1, L)` dependence rank counts for one variogram direction, channels summed."""
    names = [str(d) for d in z["dep_directions"]]
    if direction not in names:
        raise KeyError(f"direction {direction!r} not in {names}")
    return np.asarray(z["dep_hist_counts"]).sum(axis=1)[:, names.index(direction), :]


def iso_counts(z):
    """`(M+1, L)` isotropy rank counts, channels summed."""
    return np.asarray(z["iso_hist_counts"]).sum(axis=1)


def lag_labels(z):
    """One legend label per lag / distance bin, LaTeX-safe."""
    # `distance_bin` is the station network (pairs enumerated directly);
    # `grid_distance_bin` is a lat-lon grid whose strided lags were converted to km
    # and binned. Different machinery, identical axis, so identical labels.
    if str(z["dep_kind"]) in ("distance_bin", "grid_distance_bin"):
        edges = np.asarray(z["dep_bin_edges_km"], dtype=float)
        out = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            out.append(f"$>${lo:.0f} km" if not np.isfinite(hi) else f"{lo:.0f}--{hi:.0f} km")
        return out
    return [f"$h={int(h)}$" for h in z["dep_lags"]]


def emit(fig, figname, save=True, why="nothing to plot"):
    """Save the figure if SAVE is on and there is anything in it, and report where it went.

    `why` is the note printed when `save=False`, because there are now two reasons a figure
    is drawn and not written: a row with no diagnostics in it (the default) and a variant
    kept for reference only (the cross-dataset backup). Printing the wrong one is how a
    deliberately unexported figure starts looking like an empty one.
    """
    if figname in NOT_EXPORTED:
        save, why = False, "not used in the paper"
    if SAVE and save:
        path = pc.save(fig, f"calibration_{figname}.pdf")
        print(f"saved {path}")
    elif not save:
        print(f"{figname}: {why}, not saved")
    return fig


## Figure builders

One row of panels per figure, one panel per arm, so the figures can be assembled
independently in the paper. Arms with no diagnostics become annotated blank panels, and every
panel pools all of its arm's seeds. With `plot_ecdf`'s `kind="calibration"`, marginal, location and scale are three colored
lines in one panel per arm instead of three separate bar-histogram figures. `plot_ecdf_grid` stacks
those rows into one figure, datasets down and method families across -- the form the paper
uses.

In [ ]:
def _layout(figname, columns, aspect, rescale=True, n_panels=4):
    """Resolve `(columns, aspect, rc overrides, label key)` for one figure.

    Three decisions, each keyed on the thing that actually governs it:

    * **Height.** `aspect` is a fraction of the *width*, so leaving it alone would make a
      single-column figure only half as tall in absolute terms and squash the panels. It is
      rescaled so the row keeps the same physical height at either width -- pass `aspect=`
      explicitly to override. It is only the starting height, though: `plot_ecdf` then refits
      it so that every panel has the shape `PANEL_ASPECT`, four arms or five. `rescale=False` is for a
      figure whose `aspect` was *designed* at the narrow width (the cross-dataset grid
      below), where the aspect is taken as final.
    * **Fonts.** Stepped down for a narrow *figure* only (`NARROW_BELOW`), so every
      double-column figure here -- four panels or five -- carries plot_config's title, label
      and tick sizes rather than its own.
    * **Arm labels.** Long or `short`, by the width of one *panel* (`NARROW_PANEL_BELOW`):
      what overruns is a 24-character title inside a 1.35-in panel, not the figure.
    """
    cols = FIG_COLUMNS.get(figname, "double") if columns is None else columns
    width = pc.width_of(cols)
    narrow = width < NARROW_BELOW
    if rescale and narrow:
        aspect = aspect * PLOT.double_column_width / width
    key = "label" if width / n_panels >= NARROW_PANEL_BELOW else "short"
    return cols, aspect, (NARROW_RC if narrow else {}), key


def _blank_panel(ax, arm, key):
    """A panel for an arm this project has no diagnostics for.

    Keeps the title, so the column still reads as "Diag (NLL)" and the reader can see
    *what* is missing, and prints the arm's `note` where the bars would be. The frame and
    ticks are removed per-axes -- `set_xticks([])` would propagate through `sharex`/`sharey`
    and strip the tick labels off the whole row. The grid has to go too: the `science` style
    turns it on globally, and it survives the spines being hidden, leaving a floating mesh
    ruled straight through the note.
    """
    ax.set_title(arm[key])
    ax.grid(False)
    ax.text(0.5, 0.5, arm.get("note", "not available"), transform=ax.transAxes,
            ha="center", va="center", color=GUIDE, style="italic")
    # All four sides: the `science` style puts ticks on top and right as well, and those
    # outlive the spines too -- leaving a dotted rule floating where the panel edge was.
    ax.tick_params(which="both", top=False, bottom=False, left=False, right=False,
                   labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def _ecdf_series(z, kind, direction=None):
    """One `(label, color, (u, F_n(u)))` tuple per line drawn in one panel, in draw order.

    `kind="dependence"` / `"isotropy"` overlay one line per lag or distance bin, colored by
    the shared sequential ramp (`LAG_CMAP`) so the ordering reads as "further apart".
    `kind="calibration"` overlays the three PIT-based pre-ranks -- marginal, location and
    scale -- each on its own fixed color (`CALIBRATION_COLOR`). This is what lets the three
    histogram rows a dataset used to need collapse into one panel per arm: a calibrated
    diagnostic sits on the diagonal exactly as a lag's ECDF does, so the same machinery draws
    both, and only where the lines come from differs.

    Every curve goes through `ecdf_over_seeds`, so what is drawn is the per-seed median (see
    `SEED_AGGREGATION`) rather than the pooled histogram. The lag *count* and the lag
    *labels* still come off the summed arrays -- they are properties of the axis, which the
    seeds are guaranteed to agree on, not of the data.
    """
    if kind in ("dependence", "isotropy"):
        counts = iso_counts(z) if kind == "isotropy" else dep_counts(z, direction)
        names = lag_labels(z)
        colors = LAG_CMAP(np.linspace(0.12, 0.88, counts.shape[1]))
        return [(names[j], colors[j], ecdf_over_seeds(z, lag_ecdf, kind, direction, j))
                for j in range(counts.shape[1])]
    if kind == "calibration":
        return [(CALIBRATION_LABEL[obj], CALIBRATION_COLOR[obj],
                 ecdf_over_seeds(z, objective_ecdf, obj))
                for obj in CALIBRATION_OBJECTIVES]
    raise ValueError(f"unknown kind {kind!r}")


def _fit_panel_aspect(fig, ax, panel_aspect, tol=1e-3, max_iter=4, gain=1.0):
    """Set the figure height so that `ax`'s box is `panel_aspect` (height / width).

    Constrained layout hands the panels whatever height the titles, tick labels and legend
    leave over, so the box can only be set to an exact shape by measuring it: draw, compare,
    grow or shrink the figure by the difference, repeat. The overhead barely moves with the
    height, so this converges in one or two passes. The width is never touched -- it is the
    paper's column width.

    `gain` is how much figure height buys one inch of panel height, inverted: 1 for a single
    row, and the sum of the height ratios for a stacked grid, whose panels share any change
    of height between all of its rows (`plot_ecdf_grid`).
    """
    for _ in range(max_iter):
        fig.canvas.draw()
        width, height = fig.get_size_inches()
        box = ax.get_position()
        delta = panel_aspect * box.width * width - box.height * height
        if abs(delta) < tol:
            break
        fig.set_size_inches(width, height + gain * delta)


#: Headroom above the largest |u - F_n(u)| drawn, as a fraction of it, when
#: `plot_ecdf(deviation=True)` picks its symmetric y-limits. Enough that the
#: extreme curve does not touch the frame, small enough that the shape still
#: fills the panel -- which is the whole point of the deviation view.
DEVIATION_HEADROOM = 0.12


def plot_ecdf(experiment, dataset, arms, figname, kind="dependence",
              mode=None, columns=None, aspect=ROW_ASPECT, direction=None, legend_cols=None,
              deviation=False, deviation_share_y=True, panel_aspect=PANEL_ASPECT):
    """A row of ECDF panels, one per arm, with every line overlaid as a color.

    Calibration is the diagonal: above it the observed pre-rank sits too low among the
    members, below it too high. `kind` is "dependence" (needs a variogram `direction`),
    "isotropy", or "calibration" -- the pooled marginal, location and scale pre-ranks drawn
    together in one panel, which is what replaced this notebook's three separate
    bar-histogram rows per dataset. Seeds are pooled per panel, as everywhere else here.

    `panel_aspect` is the height / width of every panel box; the figure height is fitted to
    it once everything is drawn (`PANEL_ASPECT`; `None` keeps the height `aspect` gives).

    `legend_cols` forces the legend onto that many columns. **Dependence legends default to
    a single row**; every other kind adapts its column count to the figure width.

    **`deviation=True` draws `u - F_n(u)` against a flat zero line** instead of `F_n(u)`
    against the diagonal. Same curves, same data, same pooling -- only the reference is
    subtracted out, which frees the whole panel height for the departure from uniformity
    rather than spending it on the diagonal both views share. Use it to *read* a miscalibration
    whose size is a few per cent; the ECDF view stays the one that shows the curve is a CDF.

    Two things to know before comparing the two side by side:

    * **The sign is flipped relative to the diagonal reading.** A curve *above* the diagonal
      in the ECDF view has `F_n(u) > u`, so it sits *below* zero here. Above the diagonal
      means the observed pre-rank sits too low among the members; in this view that is a
      negative excursion.
    * **The y-scale is per figure, not fixed.** Limits are symmetric about zero and sized to
      the largest excursion actually drawn (`DEVIATION_HEADROOM`), so two deviation figures
      of different datasets are *not* on a common scale the way two ECDF panels are. Read the
      axis before comparing across figures.

    `deviation_share_y` is the trade-off this view forces and the reason it is a parameter.
    With one shared scale (the default, matching every other row here) the panels of a row
    stay comparable — but the scale is set by the *worst* arm, so one badly miscalibrated
    column flattens the rest into a band near zero, which is exactly the resolution the
    deviation view was meant to buy. ERA5 is the live case: `Diag`'s location and scale
    pre-ranks are near-degenerate and reach $\pm 0.5$, so `Ours` is drawn in the middle
    tenth of the axis. Pass `deviation_share_y=False` to give every panel its own symmetric
    limits and its own tick labels: each arm's shape becomes readable, and no two panels are
    on the same scale any more. Neither is the right default for every row -- pick per figure.

    The layout is resolved from `figname` so the variant matches the figure it mirrors, but
    it is saved as `<figname>_deviation` -- the ECDF PDF is never overwritten, and both can
    sit side by side while the choice between them is open.
    """
    cols, aspect, rc, key = _layout(figname, columns, aspect, n_panels=len(arms))
    handles, labels = [], []
    real = [i for i, a in enumerate(arms) if has_data(a)]
    reaches = {}                     # largest |u - F_n(u)| per panel, for symmetric limits

    with plt.rc_context(rc):
        # `sharey` has to be decided at construction: under a shared y axis a per-axes
        # `set_ylim` propagates to every panel, so the last panel set would silently win and
        # the other arms would be drawn running off their frames. Only the per-panel
        # deviation view wants independent axes; every other row keeps one shared scale.
        fig, axes = pc.subplots(1, len(arms), aspect=aspect, columns=cols,
                                sharey=deviation_share_y or not deviation)
        axes = np.atleast_1d(axes)
        for i, (ax, arm) in enumerate(zip(axes, arms)):
            if not has_data(arm):
                _blank_panel(ax, arm, key)
                continue
            z = load_pooled(experiment, dataset, arm, mode)
            if deviation:
                ax.axhline(0.0, color=GUIDE, lw=PLOT.thin_line_width, ls=":", zorder=1)
            else:
                ax.plot([0.0, 1.0], [0.0, 1.0], color=GUIDE, lw=PLOT.thin_line_width,
                        ls=":", zorder=1)
            for label, color, (u, f) in _ecdf_series(z, kind, direction):
                y = u - f if deviation else f
                # A rank ECDF can carry NaN (an all-undefined bin), and nanmax over an
                # all-NaN slice warns and returns NaN -- which would poison the limits for
                # the whole figure, not just this line.
                if deviation and np.isfinite(y).any():
                    reaches[i] = max(reaches.get(i, 0.0), float(np.nanmax(np.abs(y))))
                line, = ax.plot(u, y, color=color, lw=PLOT.line_width * 0.75,
                                label=label, zorder=2)
                if i == real[0]:
                    handles.append(line)
                    labels.append(label)
            ax.set_title(arm[key])
            ax.set_xlim(0.0, 1.0)
            if not deviation:
                ax.set_ylim(0.0, 1.0)
                # Pinned: at the five-panel rows' fitted height the auto locator drops
                # to "0, 1", so two rows of one figure family would label the axis
                # differently.
                ax.set_yticks([0.0, 0.5, 1.0])
            ax.set_xlabel("$u$")
        if deviation and real:
            # Symmetric about zero so the flat reference stays visually centred; a fallback
            # span keeps the axis sane if every curve is exactly uniform.
            def _span(r):
                return r * (1.0 + DEVIATION_HEADROOM) or 0.01
            if deviation_share_y:
                span = _span(max(reaches.values(), default=0.0))
                for ax in axes:
                    ax.set_ylim(-span, span)
            else:
                # Per-panel limits mean per-panel ticks: with sharey the tick labels live on
                # the lead axes only, and reading a panel against someone else's numbers is
                # worse than the crowding.
                for i in real:
                    axes[i].set_ylim(-_span(reaches.get(i, 0.0)), _span(reaches.get(i, 0.0)))
                    axes[i].tick_params(labelleft=True)
        if real:
            lead = axes[real[0]]
            lead.tick_params(labelleft=True)
            lead.set_ylabel("$u - F_n(u)$" if deviation else "$F_n(u)$")
        # The legend is a figure-level artist placed outside the axes, so it never covers a
        # curve; at single-column width it wraps to fewer columns instead of overrunning.
        #
        # **A dependence legend is always one row.** Its entries are an ordered ladder -- lags
        # h = 1, 2, 4, ... or near-to-far distance bins -- and the adaptive column count wraps
        # a ladder into a block, which then reads *down-then-across*: the reader has to
        # reconstruct the order rather than scan it. One line keeps it left to right.
        #
        # A nine-rung ladder (ETTh T192) on one line is the widest artist in the figure, so the
        # handles and the gaps are tightened for that case only -- otherwise the legend, not
        # the panels, would set how wide the figure has to be. `legend_cols` still wins over
        # both branches, so a caller can still force a wrap.
        if handles:
            one_row = kind == "dependence"
            ncols = legend_cols or (len(labels) if one_row
                                    else min(len(labels), 3 if rc else 8))
            fig.legend(handles, labels, loc="outside lower center", ncols=ncols,
                       handlelength=1.0 if one_row else 1.4,
                       columnspacing=0.8 if one_row else 1.1,
                       handletextpad=0.5 if one_row else 0.8)
        # Last, once every artist that takes room exists. Any panel will do: they share one
        # gridspec row, so a blank panel's box is the same as a drawn one's.
        if panel_aspect is not None:
            _fit_panel_aspect(fig, axes[0], panel_aspect)
    return emit(fig, f"{figname}_deviation" if deviation else figname, save=bool(real))


def plot_ecdf_grid(rows, figname, kind="calibration", columns=None, arm_columns=GRID_COLUMNS,
                   panel_aspect=PANEL_ASPECT):
    """Every row of `rows` stacked into one figure: datasets down, method families across.

    The single-row figures of `plot_ecdf`, one grid per `kind`, drawn with the same series,
    the same seed median and the same panel shape (`PANEL_ASPECT`). Columns are matched by
    each arm's `short` name against `arm_columns`, so a family keeps its column in every row
    and a row without it (Chol outside ETTh and EUPPBench) gets a blank panel there with
    `GRID_BLANK_NOTE`. Column titles are drawn on the top row only; x ticks and the $u$ label
    on the bottom row; y ticks in the first column.

    **The row label is the dataset**, in the place the single-row figures print $F_n(u)$
    (which the caption now has to name). It is set horizontally: a vertical "EUPPBench" is
    ~0.7 in long against rows ~0.36 in tall, and collided with its neighbours at 8 pt too --
    constrained layout ignores a y label's vertical extent, so nothing made room for it.

    **The legend depends on `kind`**:

    * `"calibration"` -- one joint legend under the grid: marginal, mean and scale are the
      same three colours in every panel.
    * `"dependence"` -- **one legend per row**, directly under it, because the lag ladder is
      per dataset (hours, 15-min steps, pixels, km bins). Each row's arms are checked to share
      one ladder, since a row legend speaks for all of its panels. The legends are placed by
      hand, not by constrained layout (see `ROW_LEGEND_GAP`): each sits in a strip of the
      gridspec below its row, and the strip is resized to the measured legend.
    """
    cols, _, rc, _ = _layout(figname, columns, ROW_ASPECT, n_panels=len(arm_columns))
    width = pc.width_of(cols)
    per_row = kind == "dependence"
    n, m = len(rows), len(arm_columns)
    step = 2 if per_row else 1                  # panel row, legend strip, panel row, ...
    handles, labels = [], []
    row_legends = []                            # (row index, legend strip, legend)
    drawn = False

    with plt.rc_context(rc):
        # Only a starting height: `_fit_panel_aspect` sets the real one below.
        fig = plt.figure(figsize=(width, n * (panel_aspect * width / m + 0.3) + 0.6),
                         layout="constrained")
        fig.get_layout_engine().set(h_pad=PLOT.layout_h_pad, w_pad=PLOT.layout_w_pad,
                                    hspace=PLOT.layout_hspace, wspace=PLOT.layout_wspace)
        gs = fig.add_gridspec(step * n, m, height_ratios=[1.0, 0.5] * n if per_row else None)
        axes = np.empty((n, m), dtype=object)
        for i in range(n):
            for j in range(m):
                lead = axes[0, 0] if (i or j) else None
                axes[i, j] = fig.add_subplot(gs[step * i, j], sharex=lead, sharey=lead)

        for i, row in enumerate(rows):
            by_short = {a["short"]: a for a in row["arms"]}
            row_handles, row_labels, ladder = [], [], None
            for j, name in enumerate(arm_columns):
                ax = axes[i, j]
                arm = by_short.get(name, dict(short=name, slug=None, seeds=[],
                                              note=GRID_BLANK_NOTE))
                if not has_data(arm):
                    _blank_panel(ax, dict(arm, short=name if i == 0 else ""), "short")
                    continue
                drawn = True
                z = load_pooled(row["experiment"], row["dataset"], arm, row["mode"])
                if per_row:
                    names = lag_labels(z)
                    if ladder is None:
                        ladder = names
                    elif names != ladder:
                        raise ValueError(
                            f"{row['label']}: {arm['slug']} is on the ladder {names}, the "
                            f"row's first arm on {ladder} -- one legend cannot label both."
                        )
                ax.plot([0.0, 1.0], [0.0, 1.0], color=GUIDE, lw=PLOT.thin_line_width,
                        ls=":", zorder=1)
                for label, color, (u, f) in _ecdf_series(z, kind, row["direction"]):
                    line, = ax.plot(u, f, color=color, lw=PLOT.line_width * 0.75,
                                    label=label, zorder=2)
                    if label not in row_labels:
                        row_handles.append(line)
                        row_labels.append(label)
                if i == 0:
                    ax.set_title(name)
                ax.set_xlim(0.0, 1.0)
                ax.set_ylim(0.0, 1.0)
                ax.set_yticks([0.0, 0.5, 1.0])
                # Per axes rather than `label_outer`: with the legend strips in the gridspec,
                # no panel row is the gridspec's last row, and `label_outer` would strip the
                # bottom row's ticks too.
                ax.tick_params(labelbottom=(i == n - 1), labelleft=(j == 0))
                if i == n - 1:
                    ax.set_xlabel("$u$")
            axes[i, 0].set_ylabel(row["label"], rotation=0, ha="right", va="center")
            if per_row and row_handles:
                strip = fig.add_subplot(gs[step * i + 1, :])
                strip.axis("off")
                leg = strip.legend(row_handles, row_labels, loc="upper center",
                                   borderaxespad=0.0, ncols=len(row_labels),
                                   handlelength=1.0, columnspacing=0.8, handletextpad=0.5)
                leg.set_in_layout(False)
                row_legends.append((i, strip, leg))
            elif not per_row and not handles:
                handles, labels = row_handles, row_labels

        if handles:
            fig.legend(handles, labels, loc="outside lower center", ncols=len(labels))
        if panel_aspect is not None:
            _fit_panel_aspect(fig, axes[0, 0], panel_aspect,
                              gain=sum(gs.get_height_ratios()), max_iter=8)
        if row_legends:
            above, below = ROW_LEGEND_GAP

            def row_bottoms():
                # Lowest decoration of each row in inches: the frame on inner rows, the
                # tick labels and $u$ on the bottom one.
                renderer = fig.canvas.get_renderer()
                return [min(ax.get_tightbbox(renderer).y0 for ax in axes[i]) / fig.dpi
                        for i, _, _ in row_legends]

            # Size every strip to hold its legend, then refit: the strips are gridspec rows,
            # so they scale with the figure and the panels have to be brought back to shape.
            height = fig.get_size_inches()[1]
            panel_h = axes[0, 0].get_position().height * height
            leg_h = max(leg.get_window_extent().height for _, _, leg in row_legends) / fig.dpi
            sep = max(b - s.get_position().y1 * height
                      for b, (_, s, _) in zip(row_bottoms(), row_legends))
            strip_h = max(above + leg_h + below - sep, 0.01)
            gs.set_height_ratios([1.0, strip_h / panel_h] * n)
            if panel_aspect is not None:
                _fit_panel_aspect(fig, axes[0, 0], panel_aspect,
                                  gain=sum(gs.get_height_ratios()), max_iter=8)
            # Anchored in figure fractions, which survive the dpi change of `pc.save`.
            fig.canvas.draw()
            height = fig.get_size_inches()[1]
            for bottom, (_, strip, leg) in zip(row_bottoms(), row_legends):
                box = strip.get_position()
                leg.set_bbox_to_anchor((box.x0 + box.width / 2, (bottom - above) / height),
                                       transform=fig.transFigure)
    return emit(fig, figname, save=drawn)


#: The pre-rank columns of the cross-dataset grid, in draw order. Only the first three are
#: drawn; `plot_cross_dataset_ecdf(dependence=True)` appends the fourth (the backup variant).
#: Dependence is last precisely because it is the one column that can be dropped without
#: leaving a hole: marginal, location and scale are three PIT pre-ranks of the same
#: closed-form predictive and read as a set, while the dependence curve is a pooled rank over
#: a per-geometry lag ladder -- a coarser question, answered properly one dataset at a time by
#: the dependence rows above.
CROSS_COLUMNS = [("marginal", "Marginal"), ("loc", "Location"), ("scale", "Scale"),
                 ("dep", "Dependence")]

#: Height of the cross-dataset grid, keyed by column count, so both variants carry *square*
#: panels. The diagonal is the null every panel here is read against, and it only looks like
#: the 45-degree line it is in a square panel -- dropping a column at fixed height would
#: widen each panel by a third and flatten the diagonal with it. So the three-column grid is
#: taller (0.95 x 3.25 in against 0.74) rather than squatter. Both numbers are measured, not
#: guessed: each gives a panel box of h/w = 1.00 at single-column width under `NARROW_RC`.
CROSS_ASPECT = {3: 0.65, 4: 0.74}


def plot_cross_dataset_ecdf(figname="cross_dataset_loc_scale", columns=None, aspect=None,
                            dependence=False, save=None):
    """3x3 ECDFs: parameterization (rows) by pre-rank (columns), every dataset overlaid.

    One line per dataset in every panel, so the question is whether a parameterization
    behaves the same way *across tasks* rather than how one task's arms compare with each
    other. Calibration is the diagonal, as everywhere else.

    Rows are the three families the paper contrasts, top to bottom: physical low-rank
    (`phys_lora_r1_kernel_score` throughout), the sample-based sampler, and ours last -- the
    reading order the tables use, with the proposed core in the row the eye ends on. The
    generative row carries EUPPBench, ERA5 (since 2026-09-07), ETTh1/ETTh2 (drawn at T96)
    and NYU-BTS (diagnosed 2026-09-21), so every row now holds all
    five datasets and `has_data` skips nothing here. Depth's sampler is the degenerate one --
    its ensemble has collapsed to a point forecast on every seed, so its curve hugs the panel
    edges instead of the diagonal. That is the arm's real behaviour and is left on the page.

    Columns are the three PIT pre-ranks, left to right: marginal, location and scale.

    **`dependence=True` appends the pooled dependence column**, which is the 3x4 grid this
    figure was until 2026-09-22. It is kept as a backup rather than deleted, because the
    curves themselves are sound -- what was wrong was the company they kept. The column
    pools every lag or distance bin into one curve per dataset (`dep_ecdf_pooled`), which it
    has to: the ladders are per-geometry -- pixel offsets on a grid, kilometres on a station
    network -- so nothing finer is well posed in a panel that overlays five datasets.
    Pooling preserves the null (each lag is uniform under calibration, so any mixture is)
    and costs the *where*: a departure no longer says at which range. Next to three
    pre-ranks of the same predictive that is a quarter of the grid spent on a coarser
    question, and the per-experiment dependence figures above answer it properly.

    So the backup does not export: `save=None` resolves to off whenever `dependence` is on,
    and an explicit `save=True` writes `<figname>_with_dependence` instead. The two variants
    share one filename stem and would otherwise overwrite each other -- with the 3x4 grid
    landing in the paper's PDF while this cell printed a cheerful "saved".

    No legend: the dataset colors are the same five used throughout the notebook, and at
    this width the strip cost more height than it bought. The proxy handles are still
    collected below, so restoring it is uncommenting one call.

    The filename still says `loc_scale` from when this was a 2x2 of location and scale. It
    is kept so an existing \\includegraphics does not break; rename it here, in FIG_COLUMNS
    and on disk together.

    `aspect` defaults per column count (`CROSS_ASPECT`) and is then taken as final rather
    than rescaled for the column width -- both variants are designed at their own width.
    """
    cols_ = CROSS_COLUMNS if dependence else CROSS_COLUMNS[:3]
    if save is None:
        save = not dependence
    if aspect is None:
        aspect = CROSS_ASPECT[len(cols_)]
    cols, aspect, rc, _ = _layout(figname, columns, aspect, rescale=False)
    rows = [("lora", "LorD"), ("generative", "SB"), ("whitened", "SCORE")]
    seen_labels = {}

    with plt.rc_context(rc):
        fig, axes = pc.subplots(len(rows), len(cols_), aspect=aspect, columns=cols,
                                sharex=True, sharey=True)
        for i, (arm_key, row_name) in enumerate(rows):
            for j, (objective, col_name) in enumerate(cols_):
                ax = axes[i, j]
                ax.plot([0.0, 1.0], [0.0, 1.0], color=GUIDE, lw=PLOT.thin_line_width,
                        ls=":", zorder=1)
                for spec in CROSS_DATASETS:
                    arm = spec[arm_key]
                    if not has_data(arm):
                        continue
                    z = load_pooled(spec["experiment"], spec["dataset"], arm, spec["mode"])
                    u, f = (ecdf_over_seeds(z, dep_ecdf_pooled, spec["dep_direction"])
                            if objective == "dep"
                            else ecdf_over_seeds(z, objective_ecdf, objective))
                    line, = ax.plot(u, f, color=DATASET_COLORS[spec["label"]],
                                    lw=PLOT.line_width * 0.75, zorder=2)
                    # Collected across *every* panel, keyed by label: a dataset can be
                    # absent from one row (no r=1 arm, no SB arm) and present in
                    # another, and keying on panel (0, 0) alone would drop it from the
                    # legend while still drawing its curve elsewhere.
                    if spec["label"] not in seen_labels:
                        seen_labels[spec["label"]] = line
                ax.set_xlim(0.0, 1.0)
                ax.set_ylim(0.0, 1.0)
                if i == 0:
                    ax.set_title(col_name)
                if j == 0:
                    ax.set_ylabel(row_name)
                if i == len(rows) - 1:
                    ax.set_xlabel("$u$")
        # Ordered by CROSS_DATASETS rather than by first appearance, so the legend order
        # matches the order the datasets are declared in whichever panel they showed up.
        labels = [d["label"] for d in CROSS_DATASETS if d["label"] in seen_labels]
        handles = [seen_labels[k] for k in labels]
        # if handles:
        #     # Five datasets do not fit on one line at this width; three per row keeps the
        #     # labels legible instead of clipping the last one.
        #     fig.legend(handles, labels, loc="outside lower center",
        #                ncols=min(len(labels), 3),
        #                handlelength=1.2, columnspacing=0.9)
    return emit(fig, f"{figname}_with_dependence" if dependence else figname,
                save=save, why="backup variant, export off")


## Paper figures — every dataset in one grid

The paper's two appendix calibration figures (Figures 11 and 12): the rows of sections 1–4 stacked, **datasets down**
(ETTh1, ETTh2, ETTm1, ETTm2, ERA5, NYU-BTS, EUPPBench — `GRID_ROWS`: the time series first,
then ERA5, depth and EUPPBench) and **method families
across** (Diag, LorD $r{=}1$, Chol, SB, SCORE — `GRID_COLUMNS`). Every panel is the same
panel its single-row figure below draws: same arms, seeds, per-seed median, variogram
direction and panel shape (`PANEL_ASPECT`).

- **The row label is the dataset**, where the single-row figures print $F_n(u)$; the caption
  has to say that the y axis is the ECDF of the PIT and that calibration is the diagonal.
  It is written horizontally because a vertical "EUPPBench" is twice as long as a row is tall.
- **Chol is blank on four rows** ("not trained"): it exists only on ETTh and EUPPBench —
  intractable on depth and ERA5, left out of the ETTm grid. A family keeps its column anyway,
  so each column is one method all the way down.
- **Column titles name the family only.** SCORE's basis differs by row — the graph Fourier
  transform with a Matérn filter on EUPPBench, the spectral diagonal everywhere else — which the caption has to carry.
- **Calibration** has one joint legend: marginal, mean and scale are the same three colours
  in every panel. **Dependence** has one legend per row, directly under it, because every
  dataset is on its own ladder — hourly lags on ETTh, 15-min steps on ETTm, pixels on depth,
  km bins on EUPPBench and ERA5. `plot_ecdf_grid` checks that all arms of a row share one
  ladder before one legend labels them.

The SB column still carries the collapses recorded in the sections below: flat lines on
ETTm (every seed) and NYU-BTS (5 of 5), which are the degenerate ensembles, not a
plotting fault.

In [ ]:
plot_ecdf_grid(GRID_ROWS, "grid_calibration", kind="calibration")

In [ ]:
plot_ecdf_grid(GRID_ROWS, "grid_dependence", kind="dependence")

## 1. Time series — ETTh1 and ETTh2 (univariate, $T=96$)

"Univariate" here is the benchmark's `features: S` mode, **not** a one-variable dataset: ETTh has
seven channels and these runs predict only `target: OT`. So `G = 1` — one pre-rank problem per
case over the forecast horizon, no pooling across channels anywhere. The correlated axis is *lead
time*, which is the axis the parameterizations differ on, so nothing is given up by the
restriction. (The multivariate runs store `G = 7` independent per-channel diagnostics; because a
Gaussian's marginal on channel $g$'s coordinates is the block $\Sigma_{gg}$, they would still say
nothing about the cross-channel blocks — so the univariate files carry the same information with
none of the pooling ambiguity.)

**Seeds 1–5 at the two reported horizons**, $T = 96$ and $T = 192$: `seq_len: 336`,
2785 test cases at $d=96$ (2689 at $d=192$).

**Five arms, one fitting objective**, reading left to right as increasing covariance generality:
Diag, LorD $r{=}1$, Chol, then SB, then SCORE — the same order as the EUPPBench
row. `phys_cholesky_kernel_score` is a full $\Sigma = LL^{\top}$, tractable here because $d$ is
96 or 192, and it belongs on the row precisely because it bounds what *any* covariance
parameterization can achieve on this task: a gap between it and SCORE is a gap in the
parameterization, not in the fitting. **No column is blank**, on either horizon.

**SCORE is the rank-free core**, `whitened_spectral_diag_s1crps_kernel_score`, the same slug as
the depth and ERA5 rows, drawn against `phys_lora_r1` with the $r{=}1$ rule constraining the
*baseline* column.

**SB does not collapse on ETTh.** Every one of its 20 seeds (two data sets, two horizons)
keeps its spread (`crps/mae` $0.73$–$0.79$, against $0.71$–$0.74$ for Diag). Its marginal curve is
still the furthest from the diagonal in every ETTh row, but it is one population, so the
per-seed median is the typical seed rather than a compromise between two modes.

**Dependence lags are hourly physical periods** — $h \in \{1,2,4,6,12,24,48,72\}$, so $h=24$ is one
day and $h=72$ three — not the dyadic default, which would give $\{1,2,4,8,16\}$ here. All arms
share that ladder; re-running any of them without `--lags 1,2,4,6,12,24,48,72` silently writes the
dyadic one and makes the panels incomparable.

$h=72$ is deliberately past the script's 25% rule-of-thumb cap (24 of 96): a lag-$h$ variogram on a
96-step horizon has $96-h$ pairs per case, so $h=72$ leaves 24 and is the noisiest curve in the
panel — but three days is the scale the ETT seasonality actually lives on. **$h=96$ is not
available**: it would have zero pairs, and `calibration.py` drops it silently via its
`h < extent` filter rather than raising. $h=84$ (12 pairs) and $h=90$ (6) are the only rungs above
72 that exist at all.

2785 test windows per dataset. Adjacent windows share 95 of their 96 horizon steps, which is why
there is no confidence band here.

In [ ]:
plot_ecdf("time_series", "etth1", TS_ARMS, "ts_etth1_calibration",
          kind="calibration", mode="univariate/T96")

In [ ]:
plot_ecdf("time_series", "etth1", TS_ARMS, "ts_etth1_dependence",
          kind="dependence", direction="axis0", mode="univariate/T96")

In [ ]:
plot_ecdf("time_series", "etth2", TS_ARMS, "ts_etth2_calibration",
          kind="calibration", mode="univariate/T96")

In [ ]:
plot_ecdf("time_series", "etth2", TS_ARMS, "ts_etth2_dependence",
          kind="dependence", direction="axis0", mode="univariate/T96")

## 1b. Time series — ETTm1 and ETTm2 (univariate, $T=96$)

The same benchmark family at 15-minute cadence: ETTm1 and ETTm2 at `features: S`
(`target: OT`), seeds 1–5. $T = 96$ is 24 hours here and
$T = 192$ is 48; the test sets are 11425 and 11329 windows, four times ETTh's.

**Four arms, not five.** Chol is not part of the ETTm grid, so this is the ETTh row without
its Chol panel — Diag, LorD, SB, SCORE — at the same kernel-score objective and
with the same slugs (`ETTM_ARMS` is `TS_ARMS` minus that one slug).

**Read the SB panel as a collapse, as on depth.** SB is a point forecast on all
20 ETTm seeds (`crps/mae` $0.99$–$1.00$), so there is no healthy seed for the median to pick: its
curves describe a degenerate ensemble, not a calibration.

**Dependence lags are 15-minute steps**, a physical ladder rather than the dyadic default:
$h \in \{1,2,4,8,12,24,48,72\}$ is 15 min, 30 min, 1 h, 2 h, 3 h, 6 h, 12 h and 18 h. That is the
ETTh row's structure — eight rungs, capped at $h=72$ so 24 pairs remain per case — on the
sub-hourly rungs the weather row uses. The legend prints steps, so $h=4$ here is one hour, which
is $h=1$ on ETTh.

In [ ]:
plot_ecdf("time_series", "ettm1", ETTM_ARMS, "ts_ettm1_calibration",
          kind="calibration", mode="univariate/T96")

In [ ]:
plot_ecdf("time_series", "ettm1", ETTM_ARMS, "ts_ettm1_dependence",
          kind="dependence", direction="axis0", mode="univariate/T96")

In [ ]:
plot_ecdf("time_series", "ettm2", ETTM_ARMS, "ts_ettm2_calibration",
          kind="calibration", mode="univariate/T96")

In [ ]:
plot_ecdf("time_series", "ettm2", ETTM_ARMS, "ts_ettm2_dependence",
          kind="dependence", direction="axis0", mode="univariate/T96")

## 2. Depth — NYU-BTS

654 test scenes of $426\times560$ ($d = 238{,}560$, the Eigen crop of the `nyu_bts_crop`
datamodule), one channel, `M = 199` draws, seeds 1–5 for every arm.

$M = 199$ is what the whole row is on: at this field size 999 draws cost ~7 min per arm against
~1.5, and the density normalization makes the panels comparable with the $M=999$ rows anyway.

The marginal histogram is read from `marg_hist_counts`, which is exact over all
$654 \times 238\,560 \approx 1.6\times10^8$ coordinates; the stored `marg_pit` array is subsampled
to 512 coordinates per scene for size, and is deliberately not used here.

Arms: **four of four, five seeds each, one objective** — `phys_diag_kernel_score` (Diag),
`phys_lora_r1_kernel_score` (LorD), `generative_kernel_score` (SB) and
`whitened_spectral_diag_s1crps_kernel_score` (SCORE), a like-for-like comparison of the
parameterizations at one objective.

**SB's diagnostics need a 48 GB card, and this is not a tuning knob.** `--draw-chunk` controls
only the sampling call; the separate `model(x)` call in `calibration.py` takes no `n_samples` and
falls back to the config's `n_eval_samples = 100`, which at $d = 238{,}560$ asks for one 11.59 GiB
tensor inside `F.conv2d`, too much for a 24 GB card at any chunk size. Lowering
`n_eval_samples` is the wrong fix: `get_mean` estimates the mean *from* the members, so fewer
members changes the estimate rather than just the memory. Budget ~22 min per seed on a 48 GB GPU.

**Read the SB panel as a collapse, not as a calibration.** Its ensemble is degenerate: all
five seeds score $\mathrm{CRPS}/\mathrm{MAE} = 0.9997$, and the CRPS of a point forecast *is* its
MAE. The marginal PIT confirms it independently — about $0.1\%$ of the mass falls in the central
$90\%$ of the panel, against $88\%$ for Diag and $45\%$ for SCORE — because the observation lies
below every draw or above every draw and almost never between them. So the curve hugs the panel
edges instead of the diagonal. It is drawn rather than suppressed: it is the trained model's real
behaviour.

One caveat the panels do not show: `phys_lora_r16_kernel_score` is diagnosed at all five seeds
too — it stays out of the row only because of the $r{=}1$ rule, and is a one-line change in
`DEPTH_ARMS` for anyone who wants the rank contrast instead.

Dependence uses the `axial_mean` direction (the mean of the two unit-axis variograms), which is the
right summary on a square-pixel grid; the diagonals sit at $h\sqrt{2}$ and belong to the isotropy
figure instead. Anisotropy contrasts axis-vs-axis and diagonal-vs-antidiagonal at equal distance,
so a diagonal ECDF means the variogram does not change with direction.

**Lag ladder** $h \in \{1,2,4,8,16,32,64\}$ pixels — the dyadic default, which at the 25% cap of
the $426$-pixel short axis (106) stops exactly at $h=64$. It is passed explicitly all the same:
`needs_refresh` keys on `schema_version` only, so a re-run without `--lags` would silently write
a different axis into a file that then looks current.

In [ ]:
plot_ecdf("depth", "nyu_bts", DEPTH_ARMS, "depth_calibration",
          kind="calibration")

In [ ]:
plot_ecdf("depth", "nyu_bts", DEPTH_ARMS, "depth_dependence",
          kind="dependence", direction=DEPTH_DEP_DIRECTION)

In [ ]:
# plot_ecdf("depth", "nyu_bts", DEPTH_ARMS, "depth_isotropy",
#           kind="isotropy")

## 3. Station postprocessing — EUPPBench

730 daily cases over 115 European stations at 72 h lead, one channel. The station network is an
*irregular* geometry with no lag structure, so the dependence pre-rank bins station pairs by
geodesic distance instead: $0$–$50$, $50$–$100$, $100$–$200$, $200$–$300$, $300$–$500$ and
$>500$ km, holding $145 / 463 / 1372 / 1497 / 2266 / 812$ of the $6555$ pairs. The three short
edges resolve the scales this experiment cares about — the backbone's message passing cuts off at
100 km — while the $300$ and $500$ km splits break up the long-distance pairs, which would
otherwise put 70% of all pairs into a single $>200$ km bin.

**The network is smaller than it looks**: the largest station separation is $954.6$ km (median
$286$, 90th percentile $525$), so there is no bin above roughly $950$ km. An edge at $1000$ km
would produce an *empty* bin, whose pre-rank is undefined for every case — the line would silently
disappear from the panel rather than error.

A pooled all-pairs variogram must never be used here: $\sum_{i<j}(x_i-x_j)^2 = d^2 s^2_x$ exactly,
so $\rho_{\mathrm{dep}}$ would collapse to a constant and give a flat histogram *by construction*.
Binning by distance is what avoids that.

Stations do not always report, so each pre-rank is restricted to that day's reporting coordinates —
exact for a Gaussian, under the assumption that missingness does not depend on the observed value.

Arms: **five, all filled** — the widest row in the notebook and the only one where the
all-kernel-score rule costs nothing. It reads left to right as increasing covariance generality
inside physical space (Diag → LorD $r{=}1$ → Chol), then SB, then SCORE, all on seeds 1–5:

- **LorD** — `phys_lora_r1_kernel_score`, at a matched objective with SCORE.
- **SB** — `generative_kernel_score`, fitted with the fair **kernel score**, the objective the
  rest of this row is held to and the estimator the reported metric uses. This panel is one of
  the places where marginal and location pre-ranks are built from **sampled ranks** rather than
  closed-form normal CDFs, which is what `load_pooled`'s `pit_estimator` guard exists to keep
  separate.
- **Chol** is `phys_cholesky_kernel_score`, an unstructured $\Sigma = LL^{\top}$. It is
  tractable here because $V = 115$, and it is on this row as the **capacity ceiling**: it is
  what a covariance model can do on this task when nothing constrains its parameterization at
  all. Read it against SCORE — see below, because the answer is not the one the panel order
  suggests.
- **SCORE** is `whitened_graph_matern_s1crps_dense_kernel_score`: graph Fourier transform with a
  Matérn filter, stage 1 CRPS, and a stage 2 that scores the predictive covariance $\Sigma_P$
  against $y - \mu$ directly. On this *localized* basis $\tilde D$ depends on the stage-2
  parameter (through the implied marginal variances), so scoring the core against a fixed
  standardized target would not be a proper score for $\Sigma_P$; the dense form uses the same
  analytic formula on the right object, at the cost of one $V = 115$ Cholesky.

**What the ceiling shows.** Chol is the best-calibrated *baseline* on this row — its
scale tail mass is 0.306 against 0.477–0.508 for Diag, LorD and SB, and its location PIT is the
flattest of the four at 0.059. And it still fans out across distance exactly as they do: mean
rank 866 below 50 km falling to 380 beyond 500 km, against SCORE at 653 → 463. So an
unstructured full covariance, fitted on the same objective with five seeds and no structural
constraint whatsoever, does **not** recover the correlation length. That is the row's main
argument: the gap between the physical parameterizations and SCORE is not a matter of
capacity, and cannot be closed by giving a physical-space model more parameters.

All arms sit on the six-bin ladder above, and `load_pooled` refuses to pool across axes: a seed
written with a different `--bin-edges-km` would be unpoolable and would mislabel the legend
without erroring.

In [ ]:
plot_ecdf("station_postprocessing", "euppbench", EUPP_ARMS, "eupp_calibration",
          kind="calibration")

In [ ]:
# The six distance bins are an ordered ladder and belong on one line, near-to-far left to
# right -- `plot_ecdf`'s default for every dependence figure.
plot_ecdf("station_postprocessing", "euppbench", EUPP_ARMS, "eupp_dependence",
          kind="dependence", direction="distance_bin")

## 4. Surface temperature — ERA5

1092 test cases on a $160\times220$ grid over Europe ($d = 35{,}200$), $0.25^\circ$ spacing over
$35$–$74.75^\circ$N and $-12.5$–$42.25^\circ$E, lead time 3, UTC-12 snapshots.

**Dependence and isotropy are binned by physical distance here, not by pixel lag** — the same
axis EUPPBench uses, reached by different machinery. On a lat-lon grid a pixel lag is not a
distance: the latitude step is a constant $27.8$ km, but the longitude step is $\cos\phi$ times
as long and runs from $22.8$ km in the south to $7.3$ km in the north, a factor of $3.1$ across
this domain. So at equal *pixel* lag the two axes sit $1.77\times$ apart in km, and the axis-vs-axis
isotropy contrast would be measuring the grid rather than the field. Binning by km fixes that:
contrasting two directions at the same *bin* is a matched-distance comparison. (The
diagonal-vs-antidiagonal contrast never had this problem — both span one latitude step and one
longitude step, so they are at identical distance whatever the latitude.)

Distances are computed **per latitude row** by the haversine formula, so no averaged $\cos\phi$
enters and the $3.1\times$ spread is resolved rather than smeared. Bins
$0$–$50$, $50$–$100$, $100$–$200$, $200$–$400$ and $400$–$800$ km are fed from the sampling ladder
$h \in \{1,2,3,4,6,8,11,16,22,32,40\}$; every bin is reached by all four directions
($159$–$462$ (lag, row) cells each), so no panel has an undefined bin. There is deliberately no
$>800$ km bin: longitude cannot reach it at high latitude, so it would be empty for `axis1` and
the isotropy contrast there would be undefined.

The dependence panel uses the `pooled` direction — the pair-weighted variogram of all four
directions within each bin, which only becomes legitimate once they share a distance axis.

**Why not the EUPPBench code path?** That one enumerates all station pairs. Here that would be
$35\,200$ coordinates $\to$ $619$ million pairs, so the strided lags are kept as the sampling
device and each (lag, row) contribution is assigned to the km it actually spans.

**Diagnosed through two datamodule overrides, and only one of them is about file locations.**
If the data path recorded in the run configs does not resolve on the machine at hand,
`--data-path` points the datamodule at the local copy. The second override is *not* a
relocation: the runs standardize with per-variable statistics computed on the **training years
1990–2014 only**, in a separate file, while the datamodule's default
(`<data_path>/era5_statistics.npy`) was computed over the whole 1990–2022 store, validation and
test years included. Getting that wrong would not be a missing file, it would be a *different
normalization*, so the diagnostics would describe a test set these checkpoints were never fitted
for. The working invocation is

```bash
python -m src.evaluation.calibration surface_temperature/era5/*/seed*/ --device cuda \\
    --data-path <era5_data_dir>/utc12/ \\
    --dm-override statistics_path=<era5_stats_dir>/era5_statistics_train1990-2014.npy \\
    --lags 1,2,3,4,6,8,11,16,22,32,40 --bin-edges-km 0,50,100,200,400,800 --num-workers 4
```

(`--dm-override KEY=VALUE` is repeatable and forwards straight to the datamodule; `--data-path`
is the single-key spelling of the same thing. The Det arms in that glob are skipped
automatically — a Dirac has $\sigma = 0$, so no PIT exists.)

**All four panels are filled with kernel-score arms**, seeds 1–5: `phys_diag_kernel_score`
(Diag), `phys_lora_r1_kernel_score` (LorD), `generative_kernel_score` (SB) and
`whitened_spectral_diag_s1crps_kernel_score` (SCORE), a two-stage fit with CRPS in stage 1 and
the kernel score in stage 2. The kernel-score LorD ranks $r \in \{2,4,16\}$ are diagnosed too;
the $r{=}1$ rule keeps them out of this figure.

Worth knowing what the objective rule changes here, because ERA5 is a dataset where both
baselines exist under *both* objectives and the difference is dramatic. At the log score,
`phys_lora_r1_nll` is catastrophically over-dispersed — its marginal PIT is a spike at $0.5$ and
its scale pre-rank collapses onto $0$. The same parameterization at the kernel score is mildly
*under*-dispersed instead and sits in the same regime as the other panels. Most of that
collapse is the objective, not the rank-1 physical parameterization, which is precisely the
confound this row avoids. The NLL siblings (`phys_diag_nll`, `phys_lora_r{1,2,4,16}_nll`) are
all diagnosed at five seeds and remain in the tree.

**Lag ladder** $h \in \{1,2,3,4,6,8,11,16,22,32,40\}$ pixels — an 11-rung geometric ladder, not
the dyadic default. It is a *sampling device* for the km bins above rather than an axis in its
own right, and it is chosen so that all four directions reach all five bins: it fills 88% of the
(direction, bin) cells against 76% for a dense $1..40$ ladder at four times the variogram cost.
Passed explicitly, because `needs_refresh` keys on `schema_version` alone and would treat a file
written on another ladder as current.

In [ ]:
plot_ecdf("surface_temperature", "era5", ERA5_ARMS, "era5_calibration",
          kind="calibration")

In [ ]:
plot_ecdf("surface_temperature", "era5", ERA5_ARMS, "era5_dependence",
          kind="dependence", direction=ERA5_DEP_DIRECTION)


In [ ]:
# plot_ecdf("surface_temperature", "era5", ERA5_ARMS, "era5_isotropy",
#           kind="isotropy")


## 5. Influenza-like illness — national_illness (multivariate)

170 test windows, 7 channels (`% WEIGHTED ILI`, `%UNWEIGHTED ILI`, `AGE 0-4`, `AGE 5-24`,
`ILITOTAL`, `NUM. OF PROVIDERS`, `OT`), **weekly** cadence, 24-week horizon. **Five seeds
(1–5)**, all diagnosed. Illness is a multivariate task at **T24 and T36**; the T36 arms are
diagnosed at the same five seeds and are a one-line switch (`mode="multivariate/T36"`), with
158 test windows and one extra dependence rung.

**All four arms are real**: Diag, LorD (`phys_lora_r1_kernel_score`), SB
(`generative_kernel_score`) and SCORE, the rank-free `whitened_spectral_diag_s1crps_kernel_score`,
the same slug the depth and ERA5 rows use, so those rows draw one comparison: SCORE against a
rank-one physical correction. The grid also carries an NLL half (`phys_diag_nll`, `phys_lora_r{1..16}_nll`),
diagnosed and on disk, which the objective rule keeps out of this figure.

**SB collapses on 2 of 5 seeds at T36 — and on none at T24.** Seeds 4 and 5 have
`crps/mae` = 0.998 and 0.999 (a Dirac's CRPS *is* its MAE); the other three sit at 0.78–0.80,
as do every seed of every other arm. This is the failure mode SB shows on all five depth
seeds. It is why the T36 panels must be read off the
**pointwise median** over seeds that `SEED_AGGREGATION` selects and never off a pooled histogram: the
five seeds are bimodal, not noisy, and an average of two point forecasts and three calibrated
ensembles describes no model that was trained. The drawn T24 row is unaffected.

**Dependence lags are weeks**, $h \in \{1,2,4,8,12,16\}$ — the dyadic default would stop at
$h=4$ under the 25% cap of a 24-step horizon, which throws away the seasonal scales this
dataset is about. $h=16$ leaves 8 pairs per case, so read the top rung as indicative. T36 adds
$h=26$ (half a year, 10 pairs per case) and keeps the other six, so the two horizons share
every rung they can.

**Only 170 test windows**, an order of magnitude fewer than ETTh's 2785 — these histograms are
visibly noisier than every other row in the notebook, and small departures from flat should not
be over-read. Five seeds help with seed noise and nothing else: the five runs score the *same*
170 windows and differ only in initialization, so the median over them averages away
initialization, not sampling error, and the caveat above still stands.

**On pooling 7 channels.** Every pre-rank is a PIT or a rank, so it is scale-free and pooling
channels of different units is not a units error. What pooling *can* do is cancel: a channel
that is under-dispersed and one that is over-dispersed average to something flat. The panels
below pool all 7 channels with no per-channel breakdown -- the same way the dependence panel
already does -- so that cancellation is possible here and is not visible on the page.

**What these panels do not test.** The pre-ranks are computed per channel, and a Gaussian's
marginal on channel $g$'s coordinates is the block $\Sigma_{gg}$ — so this row says nothing
about the *cross-channel* blocks, however the channels are aggregated afterwards. It is a
statement about per-channel calibration, not about the multivariate covariance structure.

In [ ]:
plot_ecdf("time_series", "national_illness", ILLNESS_ARMS, "illness_t24_calibration",
          kind="calibration", mode="multivariate/T24")

In [ ]:
plot_ecdf("time_series", "national_illness", ILLNESS_ARMS, "illness_t24_dependence",
          kind="dependence", direction="axis0", mode="multivariate/T24")


## 6. Weather — weather (multivariate)

21 channels at a **10-minute** cadence — note `freq: h` in the config is not the sampling rate —
so the 96-step horizon is **16 hours** and the 192-step one 32 hours, not four and eight days.
Seeds 1–5 on the reported horizons $T \in \{96, 192\}$, a filled row with illness's four arms
(`WEATHER_ARMS`). If the data path recorded in the run configs does not resolve, pass
`--data-path`; SB can be diagnosed with a smaller `eval_batch_size`, which changes memory and
nothing else.

**Dependence lags are physical periods**: $h \in \{1,3,6,12,18,36,72\}$ = 10 min, 30 min, 1 h,
2 h, 3 h, 6 h and 12 h at $T = 96$, where 24 h ($h = 144$) is past the horizon; $T = 192$ adds
$h = 144$, one day, with 48 pairs per case, so the diurnal scale is reachable there and only
there.

**Read these panels next to the log score.** Every kernel-score-fitted Gaussian arm on weather,
SCORE included, scores nll/$d$ between $+230$ and $+920$ against $+0.5$ to $+7$ for the log-score
fits: a density far too sharp somewhere. The marginal diagnostic is where that shows up as a
shape rather than a number.

**This is the row where channel pooling is most likely to mislead.** The 21 variables span
pressure, temperature, potential temperature, dewpoint, humidity, vapour pressure, air density
and wind — including `wd (deg)`, a *circular* variable in degrees for which a Gaussian is badly
misspecified and which can plausibly be miscalibrated in the opposite direction from the rest.
That is exactly the configuration in which a pooled curve can look flat while no channel is
calibrated, and the panels below have no per-channel breakdown to catch it — they pool all 21
the same way the dependence panel already does.

The per-channel caveat from section 5 applies unchanged: these are per-channel diagnostics and
say nothing about the cross-channel blocks of $\Sigma$.

In [ ]:
plot_ecdf("time_series", "weather",
          pending("time_series", "weather", WEATHER_ARMS, "multivariate/T96"),
          "weather_t96_calibration", kind="calibration", mode="multivariate/T96")

In [ ]:
plot_ecdf("time_series", "weather",
          pending("time_series", "weather", WEATHER_ARMS, "multivariate/T96"),
          "weather_t96_dependence", kind="dependence", direction="axis0",
          mode="multivariate/T96")

## 7. Across datasets — parameterization by diagnostic

This is Figure 5 of the paper. The sections above compare arms *within* one task. This one turns
the question around: does a parameterization behave the same way **across** tasks? It is a 3x3
grid — **rows** are three families (LorD, SB, and SCORE last), **columns** are the three PIT
pre-ranks (marginal, location, scale) — and every panel carries one ECDF per dataset, with
calibration on the diagonal.

**There is no dependence column.** It would have to pool every lag or distance bin into one curve
per dataset: the ladders are per-geometry — pixel offsets on a grid, kilometres on a station
network — so nothing finer is well posed in a panel that overlays five datasets. Pooling
preserves the null but costs the *where*, so a quarter of the grid would go to a coarser question
than the three columns beside it, each of which is a PIT pre-rank of the same closed-form
predictive. The per-experiment dependence figures above keep the lags separate and are where that
question belongs. The 3x4 version is kept as a backup at the end of this section and is **not**
exported.

Arms are reused from the per-experiment configs above, so this figure cannot drift from them.
The EUPPBench arms are referenced *by name* (`EUPP_LORA`, `EUPP_OURS`, `EUPP_GEN`); the others
are positional (`TS_ARMS[1]`, `DEPTH_ARMS[1]`, …), which is safe only while those lists keep
their order — a wrong-arm plot renders perfectly happily and does not raise.

**Every panel is kernel-score fitted**, so the grid compares parameterizations at a fixed
objective, exactly as the rows above do.

**Every row carries all five datasets** (ETTh1, ETTh2, NYU-BTS, EUPPBench, ERA5); `has_data`
skips nothing here.

**Read the SB row's NYU-BTS curve as a collapse, not as a calibration.** That arm's ensemble has
collapsed to a point forecast on all five seeds (see section 2), so its curve hugs the panel
edges rather than the diagonal. It is the trained model's real behaviour and is left on the page.

**There is no legend.** The five dataset colours are the same ones used throughout the notebook
and are named in the caption; at single-column width the strip cost more height than it bought,
and the panels are what the figure is for. The handles are still collected across *every* panel
— a dataset drawn in only one row would still get an entry — so restoring it is uncommenting
one call in `plot_cross_dataset_ecdf`.

**The panels stay square.** `CROSS_ASPECT` keys the height on the column count (0.95 at three
columns against 0.74 at four); a fixed height would widen every panel of the three-column grid by
a third and flatten the diagonal, and the diagonal is the null the whole figure is read against.

The figure's filename is `cross_dataset_loc_scale` for historical reasons; renaming means
changing it in `plot_cross_dataset_ecdf`, in `FIG_COLUMNS` and on disk together, so an existing
`\includegraphics` does not break.

Single-column size, so the fonts step down automatically.

In [ ]:
plot_cross_dataset_ecdf()

### Backup — the 3x4 grid, with the dependence column

The 3x4 variant with the pooled dependence column, drawn here for reference and **not** written
to `figures/`. The two variants share one filename stem, so an export would put the 3x4 grid in
the paper's PDF: `save=None` therefore resolves to *off* whenever `dependence=True`, and an
explicit `save=True` writes `..._with_dependence.pdf` instead of over the exported figure.
`save=False` below says the same thing at the call site.

Read the fourth column as the *pooled* dependence rank — every lag or distance bin summed into
one curve per dataset — which is why it is not in the exported figure.

In [ ]:
plot_cross_dataset_ecdf(dependence=True, save=False)